# Trabajo Práctico Final: Simulación de Eventos Discretos
## Estudio de la eficiencia técnica, operativa y financiera en la infraestructura de una estación de carga de vehículos eléctricos a través de la simulación de eventos discretos en CABA

**Equipo:** Carlana Rivero, Loglen, Millán, Ojeda Cabrera.

**Objetivo:** Simular, mediante la metodología evento a evento, el funcionamiento de una red de estaciones de carga de vehículos eléctricos en CABA con expansión dinámica de la infraestructura, demanda variable por franja horaria y a largo plazo, fallas y mantenimiento de cargadores, y estructura de costos dinámica. Se busca determinar los valores de las variables de control (`RC`, `TMP`, `PDCE`) que equilibren congestión, arrepentimiento, capacidad ociosa y beneficio económico.

> Las celdas marcadas **[DESACTUALIZADO — TP 4]** son copia del TP 4 y deben actualizarse al modelo del TP Final. Las marcadas **[PENDIENTE]** son esqueleto sin implementar.

In [ ]:
# [DESACTUALIZADO — TP 4] verificar la ruta de Drive del TP Final al cargar los datos.
from google.colab import drive
# Montaje de Google Drive para el acceso a los datasets históricos
drive.mount('/content/drive')

## 1. Configuración del Entorno e Importación de Librerías

Se procede con la preparación del entorno de ejecución, incluyendo la instalación de la librería `fitter` para el modelado estadístico y la importación de módulos para el procesamiento de datos, visualización y simulación estocástica.

> **[DESACTUALIZADO — TP 4]** Falta el import de `dataclasses` (estado del motor). Verificar si la versión instalada de `fitter` expone `n_jobs` en `fit()`.

In [ ]:
pip install fitter

In [ ]:
# [DESACTUALIZADO — TP 4] agregar los imports del TP Final (ver nota de arriba).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit, OptimizeWarning
from fitter import Fitter
import math
import random
import os
import time
import warnings
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
from typing import List

## 2. Ajuste de la Curva de Crecimiento del Parque Automotor Eléctrico

La demanda de carga no es estacionaria en el largo plazo: el parque de vehículos eléctricos (BEV) e híbridos enchufables (PHEV) crece de forma acelerada hasta un punto de inflexión, a partir del cual se desacelera hasta saturar. En esta sección se ajustan distintos modelos de crecimiento a la evolución observada en CABA y en otros países con procesos de adopción comparables, se elige el que la aproxima de forma más precisa y realista, y se deriva de él el factor de crecimiento variable en el tiempo que multiplica los intervalos entre arribos generados por el motor.

La propuesta plantea la curva logística `P(t) = K / (1 + A·e^(−r·t))`; esta sección valida esa elección frente a modelos alternativos.

> **[EXPLORACIÓN]** Esta sección vuelca la exploración completa: dos variantes de la serie de CABA, cinco modelos candidatos y cuatro estrategias para fijar los parámetros. Se recortará a la alternativa elegida para la versión final del TP. Las fuentes están en la carpeta `Curva Logística/` del repositorio, que se copia a `Datos/Curva Logística/` en el Drive del TP Final: `NUBE DE PUNTOS (temporal).xlsx` (planilla del equipo), `EV data by country 2026.xlsx` (IEA, *Global EV Outlook 2026*) y `bachelor_Pham_Minh_Khoa_2023.pdf` (Pham, 2023: referencia metodológica, no se lee desde el código).

### 2.1. Carga de las series históricas

Series del parque BEV/PHEV de CABA y de los países de referencia.

In [ ]:
# Fuentes: carpeta "Curva Logística" del repositorio, copiada a la carpeta de datos del TP Final en Drive.
PATH_CRECIMIENTO = "/content/drive/MyDrive/Colab Notebooks/TP Final Simu/Datos/Curva Logística/"
PLANILLA_CABA = PATH_CRECIMIENTO + "NUBE DE PUNTOS (temporal).xlsx"
# openpyxl avisa que descarta extensiones de formato de Excel que no soporta; no afecta los valores leídos.
warnings.filterwarnings("ignore", message="Unknown extension is not supported")

planilla_caba = pd.read_excel(PLANILLA_CABA, sheet_name="Tabla y Nube de Puntos (CABA)", usecols="A:I")
planilla_caba_var = pd.read_excel(PLANILLA_CABA, sheet_name="Versión % Cargables Variable (C",
                                  usecols="A:I", nrows=len(planilla_caba))
planilla_arg = pd.read_excel(PLANILLA_CABA, sheet_name="% Vehiculos Cargables (Arg)", usecols="A:F")
iea = pd.read_excel(PATH_CRECIMIENTO + "EV data by country 2026.xlsx", sheet_name="GEVO_EV_2026")

print(f"CABA: {len(planilla_caba)} trimestres | Argentina: {len(planilla_arg)} años | IEA: {len(iea)} registros")

**Magnitud ajustada.** Para CABA se ajusta el **stock de autos enchufables (BEV + PHEV) que cargan en estaciones públicas**, en vehículos: es la demanda potencial de carga, con la que escala el IA del motor. Para los países se usa el stock de autos eléctricos de la IEA (`EV stock`, `powertrain = EV`, que la IEA define como BEV + PHEV, `mode = Cars`), también en vehículos, junto con su participación en el parque (`EV stock share`) y en las ventas de autos nuevos (`EV sales share`), en %.

**Reconstrucción de la serie de CABA.** La serie se recalcula desde las columnas de datos de la planilla (autos nuevos, % de electrificados, % de enchufables y % que carga en estaciones, todos en puntos porcentuales) y se contrasta con la columna calculada `Demanda Total` de cada hoja, con la que tiene que coincidir exactamente.

**Supuestos adoptados:**
1. *Escala temporal.* La fila `Trimestre = 0` es el primer trimestre de 2020, según el eje del gráfico de la planilla ("Trimestres desde 2020"). Cada stock se ubica al cierre de su trimestre y cada participación en ventas, en el punto medio del trimestre. Para los países (datos anuales), el stock va al cierre del año y la participación en ventas, a mitad de año.
2. *Composición del mercado.* La proporción de enchufables entre los electrificados patentados en CABA es la nacional del mismo año (hoja `% Vehiculos Cargables (Arg)`), porque no hay un dato propio de CABA.
3. *Stock inicial.* El stock de enchufables anterior a 2020 se toma nulo: la planilla registra 0 % de enchufables entre los electrificados patentados hasta 2019.
4. *Sin bajas.* No se descuentan bajas del parque. El enchufable más antiguo de la serie tiene seis años y medio, y el parque contiene unos 20 años de ventas (ver 2.2).
5. *Carga pública.* El 29 % de los usuarios carga en estaciones y el resto en su domicilio (tomamos el valor para México de esta fuente: https://iea.blob.core.windows.net/assets/a9e3544b-0b12-4e15-b407-65f5c8ce1b5f/GlobalEVOutlook2024.pdf), en proporción constante en el tiempo.

**Variantes de la serie de CABA.** La planilla trae dos versiones del % de enchufables:
- `V1`: % constante (hoja `Tabla y Nube de Puntos (CABA)`). Es el cociente enchufables / electrificados del parque nacional acumulado a 2026 (18,3 %), aplicado al stock de electrificados de todo el período.
- `V2`: % variable (hoja `Versión % Cargables Variable`), aplicado al **flujo**. La proporción de enchufables en las ventas de cada año multiplica los patentamientos del trimestre y el stock es la suma acumulada: es la misma lógica con la que la hoja `% Vehiculos Cargables (Arg)` calcula el parque nacional de enchufables.

La celda siguiente verifica que `V1` y `V2` reproducen la columna `Demanda Total` de su hoja.

In [ ]:
# Proporción de enchufables (BEV + PHEV) entre los electrificados patentados en Argentina, por año.
anio_arg = planilla_arg["Año"].astype(str).str[:4].astype(int)      # "2026\n(hasta 1° semestre)" -> 2026
patentamientos_arg = planilla_arg["Patentamientos en el año"].astype(float)
prop_enchufables_arg = planilla_arg["% de enchufables (modelos BEV+PHEV)"].astype(float)
PROP_ENCHUFABLES_ANIO = dict(zip(anio_arg, prop_enchufables_arg))
# Versión constante: enchufables / electrificados en el parque nacional acumulado al 1.er semestre de 2026.
PROP_ENCHUFABLES_CTE = (patentamientos_arg * prop_enchufables_arg).sum() / patentamientos_arg.sum()

porc_carga = planilla_caba["Porcentaje que Carga en Estaciones"].unique()
assert len(porc_carga) == 1, "La planilla tiene más de un valor de % de carga en estaciones."
PORC_CARGA_ESTACIONES = float(porc_carga[0]) / 100     # la planilla lo expresa en puntos porcentuales

caba = pd.DataFrame({
    "trimestre": planilla_caba["Trimestre"].astype(int),
    "autos_nuevos": planilla_caba["Total Autos Nuevos"].astype(float),
    "pct_electrificados": planilla_caba["Porcentaje Eléctricos (+ Híbridos)"].astype(float),  # puntos porcentuales
})
caba["anio"] = 2020 + caba["trimestre"] // 4
caba["anio_stock"] = 2020 + (caba["trimestre"] + 1) / 4       # cierre del trimestre
caba["anio_ventas"] = 2020 + (caba["trimestre"] + 0.5) / 4    # punto medio del trimestre
caba["prop_enchufables"] = caba["anio"].map(PROP_ENCHUFABLES_ANIO)
caba["electrificados_nuevos"] = caba["autos_nuevos"] * caba["pct_electrificados"] / 100
caba["pct_enchufables_ventas"] = caba["pct_electrificados"] * caba["prop_enchufables"]

caba["V1"] = caba["electrificados_nuevos"].cumsum() * PROP_ENCHUFABLES_CTE * PORC_CARGA_ESTACIONES
caba["V2"] = (caba["electrificados_nuevos"] * caba["prop_enchufables"]).cumsum() * PORC_CARGA_ESTACIONES
VARIANTES_CABA = ["V1", "V2"]

# Contraste con las columnas calculadas de la planilla: tienen que coincidir trimestre a trimestre.
for variante, hoja in [("V1", planilla_caba), ("V2", planilla_caba_var)]:
    assert np.allclose(hoja["Demanda Total"], caba[variante]), f"{variante} no reproduce la 'Demanda Total' de la planilla."
    print(f"{variante} reproduce la 'Demanda Total' de la planilla (último trimestre: {caba[variante].iloc[-1]:,.1f} vehículos)")
assert np.isclose(PROP_ENCHUFABLES_CTE, planilla_caba["Porcentaje Cargables (BEV + PHEV)"].iloc[0] / 100)
print(f"Proporción constante de enchufables: {PROP_ENCHUFABLES_CTE:.4f} | % que carga en estaciones: {PORC_CARGA_ESTACIONES:.0%}")

display(caba[["trimestre", "anio_stock", "autos_nuevos", "pct_electrificados", "prop_enchufables",
              "pct_enchufables_ventas", "V1", "V2"]].round(3))

In [ ]:
# IEA: autos (BEV + PHEV), datos históricos, con nombres de países en castellano.
NOMBRES_PAISES = {"Norway": "Noruega", "Sweden": "Suecia", "Finland": "Finlandia", "Iceland": "Islandia",
                  "Denmark": "Dinamarca", "Netherlands": "Países Bajos", "China": "China",
                  "Brazil": "Brasil", "Mexico": "México", "Colombia": "Colombia", "Chile": "Chile",
                  "Costa Rica": "Costa Rica", "Uruguay": "Uruguay"}
MAGNITUDES_IEA = {"EV stock": "stock", "EV sales": "ventas", "EV stock share": "pct_parque",
                  "EV sales share": "pct_ventas"}

paises_largo = (iea[(iea["category"] == "Historical") & (iea["mode"] == "Cars") & (iea["powertrain"] == "EV")
                    & iea["parameter"].isin(list(MAGNITUDES_IEA)) & iea["region_country"].isin(list(NOMBRES_PAISES))]
                .assign(pais=lambda d: d["region_country"].map(NOMBRES_PAISES),
                        magnitud=lambda d: d["parameter"].map(MAGNITUDES_IEA))
                .rename(columns={"year": "anio", "value": "valor"})
                [["pais", "anio", "magnitud", "valor"]]
                .reset_index(drop=True))

paises = paises_largo.pivot_table(index=["pais", "anio"], columns="magnitud", values="valor").reset_index()
# Parque total y ventas totales de autos implícitos en stock / participación.
paises["parque_total"] = paises["stock"] / (paises["pct_parque"] / 100)
paises["ventas_totales"] = paises["ventas"] / (paises["pct_ventas"] / 100)
paises["parque_sobre_ventas"] = paises["parque_total"] / paises["ventas_totales"]   # años de ventas que contiene el parque

display(paises_largo.head())
display(paises.groupby("pais")["anio"].agg(["min", "max"]).T)

### 2.2. Análisis exploratorio

Evolución temporal por país y criterio de selección de los países de referencia.

In [ ]:
# Paleta categórica en orden fijo; CABA se destaca en negro en todos los gráficos.
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
COLOR_CABA = "#0b0b0b"

def formato_ejes(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(color="#e5e5e5", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend(frameon=False, fontsize=9)

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
for i, variante in enumerate(VARIANTES_CABA):
    ejes[0].plot(caba["anio_stock"], caba[variante], marker="o", markersize=3, color=PALETA[i], label=variante)
ejes[0].set_yscale("log")
formato_ejes(ejes[0], "Stock de enchufables que cargan en estaciones (CABA)", "Año", "Vehículos (escala logarítmica)")
ejes[1].plot(caba["anio_ventas"], caba["pct_electrificados"], marker="o", markersize=3, color=PALETA[0],
             label="Electrificados (HEV + PHEV + BEV)")
ejes[1].plot(caba["anio_ventas"], caba["pct_enchufables_ventas"], marker="o", markersize=3, color=PALETA[1],
             label="Enchufables (BEV + PHEV)")
formato_ejes(ejes[1], "Participación en los patentamientos de autos nuevos (CABA)", "Año", "% de los autos nuevos")
plt.tight_layout()
plt.show()

# Tasa de crecimiento anualizada del stock: 4·ln(P_t / P_(t−1)), promediada por año calendario.
tasas = np.log(caba[VARIANTES_CABA]).diff().mul(4)
tasas["anio"] = caba["anio"]
display(tasas.groupby("anio")[VARIANTES_CABA].mean().round(2).T)

**Variantes de CABA.**
- `V1` aplica a todo el período el cociente del parque de 2026 (18,3 %), cuando en 2020–2021 sólo el 1 % de los electrificados patentados eran enchufables. Sobreestima el stock inicial y su forma es la del mercado electrificado total, dominado por los híbridos convencionales. Su tasa de crecimiento cae de 2,0/año en 2020 a 0,4/año en 2023: esa desaceleración viene del arranque de la suma acumulada desde cero, no de una saturación.
- `V2` es consistente con la definición de la demanda. Crece a 0,8–1,3/año entre 2022 y 2025 y se acelera en 2026 (1,8/año), en línea con la cuota de enchufables en las ventas, que pasa de 0,02 % en 2020 a 9,1 % en el segundo trimestre de 2026. `V1` y `V2` casi coinciden al final (1 730 y 1 687) porque el cociente constante es justamente el del parque acumulado.

Se adopta **`V2`** como serie de CABA. `V1` se mantiene en 2.4 y 2.5 para comparar.

In [ ]:
PAISES_MADUROS = ["Noruega", "Suecia", "Finlandia", "Islandia", "Dinamarca", "Países Bajos", "China"]
PAISES_LATAM = ["Brasil", "México", "Colombia", "Chile", "Costa Rica", "Uruguay"]
COLOR_PAIS = {**{p: PALETA[i] for i, p in enumerate(PAISES_MADUROS)},
              **{p: PALETA[i] for i, p in enumerate(PAISES_LATAM)}}

fig, ejes = plt.subplots(2, 2, figsize=(13, 9))
for fila, (grupo, nombre_grupo) in enumerate([(PAISES_LATAM, "pares latinoamericanos"),
                                               (PAISES_MADUROS, "mercados maduros")]):
    for pais in grupo:
        datos = paises[(paises["pais"] == pais) & (paises["anio"] >= 2010)]
        ejes[fila, 0].plot(datos["anio"] + 0.5, datos["pct_ventas"], marker="o", markersize=3,
                           color=COLOR_PAIS[pais], label=pais)
        ejes[fila, 1].plot(datos["anio"] + 1, datos["pct_parque"], marker="o", markersize=3,
                           color=COLOR_PAIS[pais], label=pais)
    ejes[fila, 0].plot(caba["anio_ventas"], caba["pct_enchufables_ventas"], color=COLOR_CABA, linewidth=2.2,
                       label="CABA (trimestral)")
    formato_ejes(ejes[fila, 0], f"Enchufables en las ventas de autos nuevos: {nombre_grupo}", "Año", "% de las ventas")
    formato_ejes(ejes[fila, 1], f"Enchufables en el parque de autos: {nombre_grupo}", "Año", "% del parque")
plt.tight_layout()
plt.show()

In [ ]:
PCT_ENCHUFABLES_CABA = caba["pct_enchufables_ventas"].iloc[-1]    # último trimestre observado (2.º de 2026)

def anio_cruce(anios, valores, umbral):
    """Primer instante en que la serie alcanza el umbral (interpolación log-lineal entre observaciones); NaN si no lo alcanza."""
    anios, valores = np.asarray(anios, float), np.asarray(valores, float)
    for i in range(1, len(valores)):
        if valores[i - 1] < umbral <= valores[i] and valores[i - 1] > 0:
            frac = (np.log(umbral) - np.log(valores[i - 1])) / (np.log(valores[i]) - np.log(valores[i - 1]))
            return anios[i - 1] + frac * (anios[i] - anios[i - 1])
    return np.nan

filas = []
for pais in PAISES_MADUROS + PAISES_LATAM:
    datos = paises[paises["pais"] == pais].dropna(subset=["pct_ventas", "stock"])
    cruce = anio_cruce(datos["anio"] + 0.5, datos["pct_ventas"], PCT_ENCHUFABLES_CABA)
    if np.isnan(cruce):
        pct_parque = stock_sobre_ventas = np.nan
    else:
        # Stock y ventas anuales interpolados al instante del cruce.
        stock = np.exp(np.interp(cruce, datos["anio"] + 1, np.log(datos["stock"])))
        ventas = np.exp(np.interp(cruce, datos["anio"] + 0.5, np.log(datos["ventas"])))
        pct_parque = np.interp(cruce, datos["anio"] + 1, datos["pct_parque"])
        stock_sobre_ventas = stock / ventas
    ultimo = datos.iloc[-1]
    filas.append({"país": pais, "cruce del % de CABA": cruce, "% del parque en el cruce": pct_parque,
                  "stock / ventas anuales en el cruce (años)": stock_sobre_ventas,
                  "% ventas último año": ultimo["pct_ventas"], "último año": int(ultimo["anio"]),
                  "parque / ventas último año (años)": ultimo["parque_sobre_ventas"]})

# CABA: stock de enchufables (sin descontar la carga domiciliaria) sobre los enchufables patentados en el último año.
enchufables_ultimo_anio = (caba["electrificados_nuevos"] * caba["prop_enchufables"]).iloc[-4:].sum()
filas.append({"país": "CABA", "cruce del % de CABA": caba["anio_ventas"].iloc[-1], "% del parque en el cruce": np.nan,
              "stock / ventas anuales en el cruce (años)": caba["V2"].iloc[-1] / PORC_CARGA_ESTACIONES / enchufables_ultimo_anio,
              "% ventas último año": PCT_ENCHUFABLES_CABA, "último año": 2026, "parque / ventas último año (años)": np.nan})
etapa = pd.DataFrame(filas).set_index("país")
print(f"Participación de enchufables en las ventas de CABA (2.º trimestre de 2026): {PCT_ENCHUFABLES_CABA:.2f} %")
display(etapa.round(2))

**Etapa de adopción y países de referencia.** El indicador de etapa es la participación de los enchufables en las ventas de autos nuevos, que es la magnitud disponible tanto para CABA como para los países. CABA está en 9,1 % (segundo trimestre de 2026). Los países cruzaron ese valor en momentos muy distintos: Noruega en 2014; Países Bajos, Islandia, Suecia, Finlandia, Dinamarca y China entre 2016 y 2021; Costa Rica en 2023, Uruguay en 2024 y Colombia en 2025. Brasil (9,0 %), México (7 %) y Chile (4 %) todavía no lo alcanzaron en 2025. En el cruce, el stock equivalía a entre 0,85 y 2,6 años de ventas de enchufables; CABA tiene 1,3, dentro de ese rango.

Se usan dos grupos, con funciones distintas:
- **Pares latinoamericanos** (Brasil, México, Colombia, Chile, Costa Rica, Uruguay). Tienen un mercado comparable y una etapa similar, y sirven para comparar la etapa y para estimar el cociente parque/ventas: entre 16 y 25 años, con una mediana de 21,2. No sirven para fijar la saturación, porque están en la misma fase que CABA. En particular, México es la referencia que había propuesto el equipo, pero está *detrás* de CABA en cuota de ventas, así que su curva no muestra todavía ni la inflexión ni el techo. La excepción es Costa Rica, el único par cuya cuota de ventas ya se desacelera (10 %, 15 % y 17 % en 2023–2025), que se usa como escenario bajo.
- **Mercados maduros** (Noruega, Suecia, Finlandia, Islandia, Dinamarca, Países Bajos, China). Ya pasaron la inflexión de la cuota de ventas, así que informan el techo y la velocidad de la adopción. Suecia, Finlandia y Noruega son los países estudiados por Pham (2023).

### 2.3. Modelos de crecimiento candidatos

Modelos a contrastar con los datos. El conjunto definitivo de candidatos se fija en esta subsección.

Candidatos (todos con `t` en años desde el origen de cada serie):
- **Logística** (propuesta): simétrica respecto de su inflexión, en `P = K/2`.
- **Gompertz**: asimétrica, con la inflexión en `P = K/e` (37 % de `K`), es decir, crecimiento rápido al principio y aproximación lenta a la saturación. En Pham (2023) es el mejor modelo para varias de las series nórdicas.
- **Richards** (logística generalizada): agrega un parámetro de forma `ν` que contiene a la logística (`ν = 1`) y tiende a Gompertz (`ν → 0`). Sirve para contrastar si los datos piden una curva asimétrica.
- **Bass**: modelo de difusión con coeficientes de innovación `p` e imitación `q`; parte de `P(0) = 0` al inicio de la adopción.
- **Exponencial**: control sin saturación. Si ajusta tan bien como los modelos con techo, la serie no contiene información sobre `K`.

Criterio de ajuste: mínimos cuadrados sobre el stock en niveles, como en Pham (2023), con varios valores iniciales por modelo. `K` se acota a 1000 veces el último dato, y un `K` que termina en esa cota se interpreta como no identificable. Para comparar se usan `R²`, `MAPE` y `AIC`; el `AIC` penaliza la cantidad de parámetros, lo que permite comparar Richards (4) con los de 3 y con la exponencial (2).

In [ ]:
# Modelos candidatos. t: años desde el origen de la serie (para CABA, desde el 1/1/2020).
def modelo_logistico(t, K, A, r):
    """Logística de la propuesta: P(t) = K / (1 + A·e^(−r·t)). Inflexión en t = ln(A)/r, con P = K/2."""
    return K / (1 + A * np.exp(-r * t))

def modelo_gompertz(t, K, b, c):
    """Gompertz: P(t) = K·exp(−b·e^(−c·t)). Asimétrica: inflexión en t = ln(b)/c, con P = K/e."""
    return K * np.exp(-b * np.exp(-c * t))

def modelo_richards(t, K, A, r, nu):
    """Richards (logística generalizada): P(t) = K / (1 + A·e^(−r·t))^(1/ν). ν = 1 es la logística; ν → 0, Gompertz."""
    return K / (1 + A * np.exp(-r * t)) ** (1 / nu)

def modelo_bass(t, K, p, q):
    """Bass acumulado: P(t) = K·(1 − e^(−(p+q)·t)) / (1 + (q/p)·e^(−(p+q)·t)), con P(0) = 0 al inicio de la adopción."""
    e = np.exp(-(p + q) * t)
    return K * (1 - e) / (1 + (q / p) * e)

def modelo_exponencial(t, a, r):
    """Exponencial sin saturación (control): P(t) = a·e^(r·t)."""
    return a * np.exp(r * t)

def arranques(nombre, t, y):
    """Valores iniciales del ajuste multi-arranque: K como múltiplo del último dato y varias tasas, anclando el primer dato."""
    t0, y0, yN = t[0], y[0], y[-1]
    salida = []
    for r in (0.3, 1.0):
        if nombre == "Exponencial":
            salida.append([y0 * np.exp(-r * t0), r])
            continue
        for k in (1.5, 5, 50):
            K = k * yN
            if nombre == "Logística":
                salida.append([K, (K / y0 - 1) * np.exp(r * t0), r])
            elif nombre == "Gompertz":
                salida.append([K, np.log(K / y0) * np.exp(r * t0), r])
            elif nombre == "Richards":
                salida += [[K, (K / y0 - 1) * np.exp(r * t0), r, 1.0], [K, (K / y0 - 1) * np.exp(r * t0), r, 0.3]]
            elif nombre == "Bass":
                salida.append([K, 0.01, r])
    return salida

COTA_K = 1000   # K se acota a 1000 veces el último dato: un K que queda en la cota no es identificable con esa serie

MODELOS = {  # nombre: (función, cotas inferiores, cotas superiores; la de K se fija en cada serie)
    "Logística": (modelo_logistico, [0, 0, 0], [np.inf, np.inf, 10]),
    "Gompertz": (modelo_gompertz, [0, 0, 0], [np.inf, np.inf, 10]),
    "Richards": (modelo_richards, [0, 0, 0, 1e-3], [np.inf, np.inf, 10, 50]),
    "Bass": (modelo_bass, [0, 1e-7, 0], [np.inf, 1, 10]),
    "Exponencial": (modelo_exponencial, [0, 0], [np.inf, 10]),
}

def ajustar(nombre, t, y, sigma=None, serie=""):
    """Mínimos cuadrados con varios arranques; devuelve (parámetros, covarianza) del arranque de menor error."""
    funcion, inf, sup = MODELOS[nombre]
    if nombre != "Exponencial":
        sup = [COTA_K * y[-1]] + sup[1:]
    mejor = None
    for p0 in arranques(nombre, t, y):
        p0 = np.clip(p0, inf, sup)
        with warnings.catch_warnings(), np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            warnings.simplefilter("ignore", OptimizeWarning)
            try:
                params, cov = curve_fit(funcion, t, y, p0=p0, sigma=sigma, bounds=(inf, sup),
                                         x_scale="jac", maxfev=2000)
            except RuntimeError:      # el arranque no convergió: se prueba el siguiente
                continue
        sse = np.sum(((y - funcion(t, *params)) / (1 if sigma is None else sigma)) ** 2)
        if np.isfinite(sse) and (mejor is None or sse < mejor[2]):
            mejor = (params, cov, sse)
    if mejor is None:
        raise RuntimeError(f"{nombre} sobre {serie}: ningún arranque convergió.")
    return mejor[0], mejor[1]

def metricas(y, y_ajustado, n_params):
    residuos = y - y_ajustado
    sse = np.sum(residuos ** 2)
    n = len(y)
    return {"R²": 1 - sse / np.sum((y - y.mean()) ** 2),
            "MAPE (%)": 100 * np.mean(np.abs(residuos / y)),
            "AIC": n * np.log(sse / n) + 2 * n_params}

### 2.4. Ajuste y comparación de bondad

Ajuste de cada modelo a cada serie y comparación según métricas de bondad de ajuste y plausibilidad de los parámetros obtenidos (en particular, la capacidad de saturación `K`).

In [ ]:
# Series a ajustar: stock en vehículos, t en años desde el inicio de la adopción registrada (P = 0 en t = 0).
series = {}
for pais in PAISES_MADUROS + PAISES_LATAM:
    datos = paises[(paises["pais"] == pais) & (paises["stock"] > 0)]
    series[pais] = ((datos["anio"] + 1 - datos["anio"].min()).to_numpy(float), datos["stock"].to_numpy(float))
for variante in VARIANTES_CABA:
    series[f"CABA {variante}"] = ((caba["anio_stock"] - 2020).to_numpy(), caba[variante].to_numpy())

def ajustar_serie(serie):
    """Ajusta los cinco modelos a una serie; devuelve las filas de la tabla comparativa y los parámetros por modelo."""
    t, y = series[serie]
    filas, parametros = [], {}
    for nombre, (funcion, _, _) in MODELOS.items():
        params, cov = ajustar(nombre, t, y, serie=serie)
        parametros[nombre] = params
        fila = {"serie": serie, "modelo": nombre, **metricas(y, funcion(t, *params), len(params))}
        if nombre != "Exponencial":
            error_K = np.sqrt(cov[0, 0]) if np.isfinite(cov[0, 0]) else np.inf
            fila.update({"K / último dato": params[0] / y[-1], "error est. de K (%)": 100 * error_K / params[0],
                         "K en la cota": bool(np.isclose(params[0], COTA_K * y[-1]))})
        filas.append(fila)
    return filas, parametros

# Las series son independientes: un proceso por vCPU. El ajuste es determinista, así que el resultado
# no depende de la cantidad de procesos. Con "fork" los procesos heredan las series y las funciones del notebook.
inicio = time.perf_counter()
with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=multiprocessing.get_context("fork")) as ejecutor:
    resultados = dict(zip(series, ejecutor.map(ajustar_serie, series)))
filas = [fila for filas_serie, _ in resultados.values() for fila in filas_serie]
ajustes = {(serie, nombre): params for serie, (_, por_modelo) in resultados.items() for nombre, params in por_modelo.items()}
print(f"{len(filas)} ajustes en {time.perf_counter() - inicio:.1f} s con {os.cpu_count()} procesos")

comparacion = pd.DataFrame(filas)
comparacion["ΔAIC"] = comparacion["AIC"] - comparacion.groupby("serie")["AIC"].transform("min")
orden_series = list(series)
tabla_aic = comparacion.pivot(index="serie", columns="modelo", values="ΔAIC").loc[orden_series, list(MODELOS)]
tabla_K = comparacion.pivot(index="serie", columns="modelo", values="K / último dato").loc[orden_series, list(MODELOS)[:-1]]
tabla_errK = comparacion.pivot(index="serie", columns="modelo", values="error est. de K (%)").loc[orden_series, list(MODELOS)[:-1]]
tabla_mape = comparacion.pivot(index="serie", columns="modelo", values="MAPE (%)").loc[orden_series, list(MODELOS)]
print("ΔAIC respecto del mejor modelo de cada serie (0 = mejor):")
display(tabla_aic.round(1))
print("MAPE (%):")
display(tabla_mape.round(1))
en_cota = (comparacion.pivot(index="serie", columns="modelo", values="K en la cota")
           .loc[orden_series, list(MODELOS)[:-1]].astype(bool))
print("K estimado / último dato observado ('cota': K terminó en 1000 veces el último dato):")
display(tabla_K.map(lambda v: f"{v:.3g}").mask(en_cota, "cota"))
print("Error estándar de K relativo a K (%) ('degenerado': covarianza singular, con otro parámetro en su cota):")
display(tabla_errK.map(lambda v: f"{v:.3g}").mask(tabla_errK < 1e-6, "degenerado").mask(en_cota, "—"))
print("Parámetro de forma ν de Richards (1 = logística; → 0, Gompertz; > 1, aproximación a K más abrupta):")
nu = pd.Series({serie: ajustes[(serie, "Richards")][3] for serie in orden_series})
display(nu.map(lambda v: f"{v:.2f}").mask(en_cota["Richards"], "cota").to_frame("ν").T)

In [ ]:
# Datos y curvas ajustadas, extrapoladas hasta 2035, en series representativas de cada grupo.
SERIES_GRAFICO = ["Noruega", "Suecia", "Finlandia", "China", "Costa Rica", "México", "Brasil", "CABA V2"]
fig, ejes = plt.subplots(2, 4, figsize=(16, 7.5))
for ax, serie in zip(ejes.flat, SERIES_GRAFICO):
    t, y = series[serie]
    origen = 2020 if serie.startswith("CABA") else paises.loc[(paises["pais"] == serie) & (paises["stock"] > 0), "anio"].min()
    t_ext = np.linspace(t[0], 2035 - origen, 300)
    for i, (nombre, (funcion, _, _)) in enumerate(MODELOS.items()):
        with np.errstate(over="ignore"):
            ax.plot(origen + t_ext, funcion(t_ext, *ajustes[(serie, nombre)]), color=PALETA[i], linewidth=1.6, label=nombre)
    ax.plot(origen + t, y, "o", color=COLOR_CABA, markersize=3.5, label="Datos")
    ax.set_ylim(0, 3 * y.max())
    formato_ejes(ax, serie, "Año", "Stock (vehículos)")
    ax.legend().set_visible(ax is ejes.flat[0])
plt.tight_layout()
plt.show()

**Elección del modelo.**
- *Series que ya pasaron la inflexión* (las nórdicas, Países Bajos, China, Costa Rica): `K` es identificable, con error estándar menor al 25 % para la logística. La logística y Bass dan ajustes prácticamente iguales (Bass con `p ≈ 0` se reduce a una logística) y quedan a ΔAIC ≤ 2 del mejor modelo en Noruega, Suecia e Islandia. Richards es el mejor en varias series, pero su parámetro de forma es inestable. En las series nórdicas da `ν` entre 0 y 0,9: una asimetría hacia Gompertz, con crecimiento rápido y saturación lenta. En Países Bajos, Brasil, Chile, Costa Rica y China, en cambio, da `ν > 1`, una saturación abrupta pegada al último dato (`K` = 1,1–1,6 veces el último valor). Eso no es plausible cuando la cuota de ventas sigue creciendo.
- *Por MAPE*, el criterio que privilegia Pham (2023), Gompertz y Bass ganan con más frecuencia. La logística ajustada en niveles reproduce los últimos años pero no la etapa temprana: su MAPE es de 1 085 % en Suecia y de 252 % en Islandia.
- *Series en fase de aceleración* (México, Colombia, Uruguay y las dos variantes de CABA): la exponencial es la mejor por AIC. Los modelos con techo dejan `K` en la cota (1000 veces el último dato) o con un error que lo vuelve indeterminado: la logística de Uruguay da `K` = 101 veces el último dato, con un error estándar del 979 %. El ΔAIC ≈ 2 de la logística es exactamente la penalización por el parámetro extra: los datos no contienen información sobre la saturación. Por eso la serie de CABA, por sí sola, no permite estimar `K`.

Se mantiene la **logística** de la propuesta, que ajusta tan bien como cualquier alternativa donde la saturación es observable. Se conserva **Gompertz** como alternativa: las series maduras muestran asimetría (`ν < 1`) y Gompertz es la mejor por MAPE en varias de ellas, en línea con Pham (2023). Los parámetros de CABA se obtienen en 2.5 con información externa.

### 2.5. Parámetros para CABA

Obtención de los parámetros del modelo elegido para CABA, a partir de su serie propia y de las características observadas en los países de referencia. Se descuenta la proporción de usuarios que cargan en su domicilio.

La serie de CABA está en plena fase de aceleración (2.4), así que `K` no se puede obtener de ella sola. Se exploran cuatro estrategias para fijar los parámetros, que difieren en qué información externa usan:

| Estrategia | `K` | `r` | Posición (`A`) |
|---|---|---|---|
| A. Ajuste libre | serie de CABA | serie de CABA | serie de CABA |
| B. Potencial de mercado (Pham, 2023) | parque estimado × potencial de saturación × % carga pública | serie de CABA | serie de CABA |
| C. Analogía por etapa de adopción | `K / P` del país de referencia en la misma etapa | país de referencia | distancia cruce–inflexión del país |
| D. Proyección por cuota de ventas | ajuste a la proyección | ajuste a la proyección | anclada al stock observado |

Las cuatro se evalúan con los mismos criterios, todos sobre el horizonte de simulación a partir de `T = 0` (1/7/2026):
- **Crecimiento** del stock a 1, 3, 5 y 10 años: es la inversa del factor que multiplicará al IA.
- **Cuota implícita**: el % de los autos nuevos que tendría que ser enchufable para producir la pendiente de la curva, `dP/dt / (ventas anuales × 29 %)`. En `T = 0` debería coincidir con el 9,1 % observado en el último trimestre, y nunca puede superar el 100 %. Esa es la **cota física**: el stock no puede crecer más rápido que si todos los autos nuevos fueran enchufables.

Supuestos de esta subsección:
- `T = 0` del motor es el 1/7/2026, cierre del último trimestre observado.
- La ventana de proyección es de 10 años, que debe cubrir el horizonte `TF` de la simulación.
- Las ventas de autos nuevos de CABA se mantienen en el nivel de los últimos cuatro trimestres.
- El parque de CABA no está en las fuentes: se estima como ventas anuales × cociente parque/ventas, con la mediana de los pares latinoamericanos en 2025 según la IEA.

In [ ]:
T_INICIO = 6.5                  # T = 0 del motor: 1/7/2026 (años desde el 1/1/2020), cierre del último trimestre observado
HORIZONTE_ANIOS = 10            # ventana de proyección y de ajuste: supuesto, debe cubrir el horizonte TF de la simulación
P_INICIO = caba["V2"].iloc[-1]  # stock observado en T = 0

VENTAS_ANUALES_CABA = caba["autos_nuevos"].iloc[-4:].sum()                  # últimos cuatro trimestres
COTA_CRECIMIENTO = VENTAS_ANUALES_CABA * PORC_CARGA_ESTACIONES             # vehículos/año con el 100 % de los autos nuevos enchufables
PARQUE_SOBRE_VENTAS = paises.loc[paises["pais"].isin(PAISES_LATAM) & (paises["anio"] == 2025), "parque_sobre_ventas"].median()
PARQUE_CABA_EST = VENTAS_ANUALES_CABA * PARQUE_SOBRE_VENTAS

print(f"Stock en T = 0: {P_INICIO:,.0f} vehículos | ventas anuales de autos nuevos: {VENTAS_ANUALES_CABA:,.0f}")
print(f"Cota física de crecimiento: {COTA_CRECIMIENTO:,.0f} vehículos/año")
print(f"Parque / ventas (mediana latinoamericana 2025): {PARQUE_SOBRE_VENTAS:.1f} años -> parque estimado de CABA: {PARQUE_CABA_EST:,.0f} autos")

def evaluar_curva(estrategia, P):
    """Resume una curva P(t) de CABA (t en años desde 2020) sobre el horizonte, a partir de T = 0."""
    t = np.linspace(T_INICIO, T_INICIO + HORIZONTE_ANIOS, 1201)
    with np.errstate(over="ignore"):
        p = P(t)
    pendiente = np.gradient(p, t)
    return {"estrategia": estrategia,
            "P(T=0) / observado": p[0] / P_INICIO,
            **{f"× a {a} años": P(T_INICIO + a) / p[0] for a in (1, 3, 5, 10)},
            "cuota implícita en T=0 (%)": 100 * pendiente[0] / COTA_CRECIMIENTO,
            "cuota implícita máx. (%)": 100 * pendiente.max() / COTA_CRECIMIENTO}

evaluaciones = {}   # estrategia: (función P(t), parámetros (K, A, r) si es logística)

#### 2.5.1. Estrategia A: ajuste libre sobre la serie de CABA

In [ ]:
# Estrategia A: ajuste libre sobre la serie propia de CABA (los ajustes de 2.4).
for variante in VARIANTES_CABA:
    K, A, r = ajustes[(f"CABA {variante}", "Logística")]
    evaluaciones[f"A · libre {variante}"] = (lambda t, K=K, A=A, r=r: modelo_logistico(t, K, A, r), (K, A, r))
    print(f"{variante}: K = {K:,.3g} ({K / caba[variante].iloc[-1]:,.3g} veces el último dato), r = {r:.3f}/año")
display(pd.DataFrame([evaluar_curva(e, P) for e, (P, _) in evaluaciones.items() if e.startswith("A")])
        .set_index("estrategia").map(lambda v: f"{v:.3g}"))

En las dos variantes `K` termina en la cota y `r` coincide con la tasa de crecimiento exponencial (0,53/año en `V1` y 1,47 en `V2`). Las curvas resultantes son exponenciales: multiplican el stock por 166 (`V1`) y por más de 1 000 (`V2`) en 10 años y requieren cuotas de enchufables de hasta 490 % (`V1`) y más de 2 500 % (`V2`) de las ventas. **Se descarta.**

#### 2.5.2. Estrategia B: `K` exógeno como potencial de mercado (Pham, 2023)

Pham (2023) fija el potencial de mercado como el 30 % del parque (iniciativa EV30@30) y ajusta sólo la tasa y la inflexión. Acá, `K = parque estimado × potencial de saturación × 29 %`, con potenciales de saturación (PS) del 30 % y del 50 %, y `A` y `r` ajustados sobre cada variante.

In [ ]:
# Estrategia B: K exógeno = parque estimado × potencial de saturación × % que carga en estaciones; A y r se ajustan.
POTENCIALES_SATURACION = [0.30, 0.50]
t_caba = (caba["anio_stock"] - 2020).to_numpy()
for potencial in POTENCIALES_SATURACION:
    K = PARQUE_CABA_EST * potencial * PORC_CARGA_ESTACIONES
    for variante in VARIANTES_CABA:
        y = caba[variante].to_numpy()
        (A, r), _ = curve_fit(lambda t, A, r: modelo_logistico(t, K, A, r), t_caba, y,
                              p0=[K / y[0], 1.0], bounds=([0, 0], [np.inf, 10]), maxfev=20000)
        evaluaciones[f"B · PS {potencial:.0%} {variante}"] = (lambda t, K=K, A=A, r=r: modelo_logistico(t, K, A, r), (K, A, r))

resumen_B = []
for estrategia, (P, (K, A, r)) in evaluaciones.items():
    if estrategia.startswith("B"):
        resumen_B.append({"estrategia": estrategia, "K": K, "r (1/año)": r, "inflexión (año)": 2020 + np.log(A) / r,
                          "pendiente máx. / cota": r * K / 4 / COTA_CRECIMIENTO})
display(pd.DataFrame(resumen_B).set_index("estrategia").round(3))
display(pd.DataFrame([evaluar_curva(e, P) for e, (P, _) in evaluaciones.items() if e.startswith("B")])
        .set_index("estrategia").round(3))

- Con `V2`, `r` es la tasa de la fase temprana (1,48/año). La pendiente máxima llega a 2,35 veces la cota física: la curva requiere que el 235 % de los autos nuevos sea enchufable. **Es físicamente imposible.**
- Con `V1`, `r = 0,53`/año, y con PS = 30 % la curva respeta la cota (84 %). Pero la cuota implícita en `T = 0` es 3,5 %, contra el 9,1 % observado: la curva hereda la velocidad del mercado electrificado total, no la de los enchufables. Con PS = 50 % ya supera la cota (140 %).
- En todos los casos `K` depende de un potencial de saturación que es una meta de política, sin respaldo empírico para CABA.

Se descarta como base. `B · PS 30% V1` queda como referencia del enfoque de Pham (2023).

#### 2.5.3. Estrategia C: analogía con los países que ya pasaron la etapa de CABA

Para cada mercado maduro, y para Costa Rica, se toma la logística ajustada en 2.4 y se ubica el instante en que su cuota de ventas cruzó el 9,1 % de CABA. De ahí se trasladan a CABA tres cosas: el cociente `K / P(cruce)`, la tasa `r` y la distancia entre el cruce y la inflexión. Se reporta también la mediana de los países.

In [ ]:
# Estrategia C: analogía con los países que ya pasaron la etapa actual de CABA.
# Se toma su logística ajustada (2.4), se ubica el cruce del % de ventas de CABA y se trasladan r, K/P(cruce) y la
# distancia entre el cruce y la inflexión. CABA cruza ese % en el 2.º trimestre de 2026, con el stock P_INICIO.
CRUCE_CABA = caba["anio_ventas"].iloc[-1]
filas = []
for pais in PAISES_MADUROS + ["Costa Rica"]:
    K_p, A_p, r_p = ajustes[(pais, "Logística")]
    origen = paises.loc[(paises["pais"] == pais) & (paises["stock"] > 0), "anio"].min()
    cruce = etapa.loc[pais, "cruce del % de CABA"]
    P_cruce = modelo_logistico(cruce - origen, K_p, A_p, r_p)
    inflexion = origen + np.log(A_p) / r_p
    K = P_INICIO * K_p / P_cruce
    inflexion_caba = CRUCE_CABA + (inflexion - cruce)
    A = np.exp(r_p * (inflexion_caba - 2020))
    filas.append({"país": pais, "cruce": cruce, "inflexión": inflexion, "K / P(cruce)": K_p / P_cruce,
                  "r (1/año)": r_p, "K CABA": K, "inflexión CABA": inflexion_caba, "A CABA": A})
analogias = pd.DataFrame(filas).set_index("país")
mediana = analogias[["K / P(cruce)", "r (1/año)"]].median()
inflexion_med = CRUCE_CABA + (analogias["inflexión"] - analogias["cruce"]).median()
analogias.loc["Mediana", ["K / P(cruce)", "r (1/año)", "inflexión CABA"]] = [*mediana, inflexion_med]
analogias.loc["Mediana", "K CABA"] = P_INICIO * mediana["K / P(cruce)"]
analogias.loc["Mediana", "A CABA"] = np.exp(mediana["r (1/año)"] * (inflexion_med - 2020))
display(analogias.round(3))

for pais in ["Noruega", "Mediana", "Costa Rica"]:
    K, A, r = analogias.loc[pais, ["K CABA", "A CABA", "r (1/año)"]]
    evaluaciones[f"C · analogía {pais}"] = (lambda t, K=K, A=A, r=r: modelo_logistico(t, K, A, r), (K, A, r))
display(pd.DataFrame([evaluar_curva(e, P) for e, (P, _) in evaluaciones.items() if e.startswith("C")])
        .set_index("estrategia").round(3))

- `K / P(cruce)` varía entre 8,7 (Costa Rica) y 57 (Países Bajos), con mediana 26,7. `r` varía entre 0,35 y 0,92/año (mediana 0,56) y la inflexión de CABA quedaría entre 2029 y 2038 (mediana 2032). El crecimiento a 10 años va de ×8 a ×33 según el país elegido.
- Todas las curvas respetan la cota física (cuota implícita máxima de 14 a 40 %). Pero en `T = 0` implican una cuota de 3,5–6,3 %, por debajo del 9,1 % observado: subestiman el ritmo actual de CABA. La razón no es la etapa, porque la profundidad del stock de CABA (1,3 años de ventas) es comparable a la de los países. Es la forma: la logística simétrica ajustada en niveles a los países subestima su propio crecimiento temprano, y al trasladarla arrastra ese sesgo.

Se descarta como base.

#### 2.5.4. Estrategia D: proyección del stock a partir de la cuota de ventas

En lugar de extrapolar el stock, se proyecta la variable que efectivamente sigue una curva en "S" en todos los mercados maduros, la **cuota de enchufables en las ventas** `s(t)`, que tiene un techo natural del 100 %. El stock se reconstruye por acumulación:

`P(t + ¼) = P(t) + ventas trimestrales × s(t) × 29 %`

`s(t)` es una logística con el techo `S_MAX` y la velocidad `r_s` de los países de referencia, anclada al 9,1 % observado en el último trimestre. Escenarios:
- **base**: la mediana de los mercados maduros;
- **alto**: Noruega, el único con techo cercano al 100 %;
- **bajo**: Costa Rica, el único par latinoamericano cuya cuota ya se desacelera.

Sobre esa proyección se ajustan la logística y Gompertz, ancladas al stock observado en `T = 0` y con error relativo, para que el primer año pese lo mismo que el décimo. Esta estrategia usa todos los datos propios de CABA (stock, cuota y ventas) y toma de los países sólo lo que CABA no puede aportar: el techo y la velocidad de la cuota.

In [ ]:
# Estrategia D, paso 1: logística sobre el % de enchufables en las ventas de los países de referencia.
# Techo S_MAX acotado a 100 %; t en años desde 2000 y datos a mitad de año.
def logistica_cuota(t, S_MAX, t_inf, r):
    return S_MAX / (1 + np.exp(-r * (t - t_inf)))

filas = []
for pais in PAISES_MADUROS + ["Costa Rica"]:
    datos = paises[(paises["pais"] == pais) & (paises["pct_ventas"] > 0)]
    t, y = (datos["anio"] + 0.5 - 2000).to_numpy(float), datos["pct_ventas"].to_numpy(float)
    mejor = None
    for S0 in (y.max() * 1.1, 60, 100):
        for r0 in (0.3, 0.7, 1.2):
            try:
                params, cov = curve_fit(logistica_cuota, t, y, p0=[min(S0, 100), t[np.argmax(y > y.max() / 2)], r0],
                                        bounds=([0, 0, 0], [100, 60, 5]), maxfev=20000)
            except RuntimeError:      # el arranque no convergió: se prueba el siguiente
                continue
            sse = np.sum((y - logistica_cuota(t, *params)) ** 2)
            if mejor is None or sse < mejor[1]:
                mejor = (params, sse, np.sqrt(np.diag(cov)))
    (S_MAX, t_inf, r), _, err = mejor
    filas.append({"país": pais, "S_MAX (%)": S_MAX, "error S_MAX": err[0], "inflexión (año)": 2000 + t_inf,
                  "r_s (1/año)": r, "% ventas último año": y[-1]})
cuotas_ref = pd.DataFrame(filas).set_index("país")
cuotas_ref.loc["Mediana maduros", ["S_MAX (%)", "r_s (1/año)"]] = cuotas_ref.loc[PAISES_MADUROS, ["S_MAX (%)", "r_s (1/año)"]].median()
display(cuotas_ref.round(2))

ESCENARIOS_CUOTA = {"base": "Mediana maduros", "alto": "Noruega", "bajo": "Costa Rica"}

In [ ]:
# Estrategia D, paso 2: proyección del stock de CABA acumulando los enchufables patentados.
# s(t) es logística con el techo y la velocidad del escenario, anclada al % observado en el último trimestre.
def proyectar_stock(S_MAX, r_s, anios=HORIZONTE_ANIOS):
    t_inf = caba["anio_ventas"].iloc[-1] + np.log(S_MAX / PCT_ENCHUFABLES_CABA - 1) / r_s
    trimestres = np.arange(1, 4 * anios + 1)
    cuota = logistica_cuota(2026.5 + (trimestres - 0.5) / 4, S_MAX, t_inf, r_s)    # % a mitad de cada trimestre
    stock = P_INICIO + np.cumsum(VENTAS_ANUALES_CABA / 4 * cuota / 100 * PORC_CARGA_ESTACIONES)
    return T_INICIO + trimestres / 4, stock, t_inf

def logistica_anclada(t, K, r):
    """Logística que pasa por P_INICIO en T_INICIO: A queda determinado por K y r."""
    return modelo_logistico(t, K, (K / P_INICIO - 1) * np.exp(r * T_INICIO), r)

def gompertz_anclada(t, K, c):
    """Gompertz que pasa por P_INICIO en T_INICIO: b queda determinado por K y c."""
    return modelo_gompertz(t, K, np.log(K / P_INICIO) * np.exp(c * T_INICIO), c)

proyecciones = {}
for escenario, referencia in ESCENARIOS_CUOTA.items():
    S_MAX, r_s = cuotas_ref.loc[referencia, ["S_MAX (%)", "r_s (1/año)"]]
    t_proy, stock_proy, _ = proyectar_stock(S_MAX, r_s)
    proyecciones[escenario] = (t_proy, stock_proy)
    t_aj, y_aj = np.r_[T_INICIO, t_proy], np.r_[P_INICIO, stock_proy]
    evaluaciones[f"D · proyección {escenario}"] = (lambda t, x=t_aj, y=y_aj: np.interp(t, x, y), None)
    # Ajuste con error relativo (sigma = y): pesa igual el primer año que el décimo.
    for forma, funcion in [("logística", logistica_anclada), ("Gompertz", gompertz_anclada)]:
        (K, r), _ = curve_fit(funcion, t_aj, y_aj, p0=[2 * y_aj[-1], 0.5], sigma=y_aj,
                              bounds=([y_aj[-1], 0], [np.inf, 10]), maxfev=20000)
        if forma == "logística":
            params = (K, (K / P_INICIO - 1) * np.exp(r * T_INICIO), r)
        else:
            params = (K, np.log(K / P_INICIO) * np.exp(r * T_INICIO), r)
        evaluaciones[f"D · {forma} {escenario}"] = (lambda t, f=funcion, K=K, r=r: f(t, K, r), params)
        error = {a: funcion(T_INICIO + a, K, r) / np.interp(T_INICIO + a, t_aj, y_aj) - 1 for a in (1, 3, 5, 10)}
        print(f"{escenario:5s} {forma:9s}: K = {K:9,.0f}  r = {r:.3f}/año  error vs. proyección: "
              + "  ".join(f"{a} a: {e:+.0%}" for a, e in error.items()))

display(pd.DataFrame([evaluar_curva(e, P) for e, (P, _) in evaluaciones.items() if e.startswith("D")])
        .set_index("estrategia").round(3))

- **Cuotas de referencia.** `S_MAX` va de 57 % (Finlandia) a 99 % (Noruega), con mediana 63,7 %, y `r_s` de 0,38 a 1,13/año, con mediana 0,81. En Costa Rica, `S_MAX` = 21 %.
- **Proyección base.** El stock se multiplica por 3,0 al año, 11,7 a los 3 años, 26,4 a los 5 y 70,6 a los 10. La cuota implícita arranca en 10,8 %, consistente con el 9,1 % observado y su tendencia, y nunca supera el techo del 63,7 %.
- **Forma de la curva.** La logística anclada no reproduce la proyección en los primeros años: el error es de −29 % al año y −25 % a los 3 años, y su cuota implícita en `T = 0` es 5,3 %. Gompertz la sigue con errores de entre −6 % y +2 % en todo el horizonte (cuota implícita en `T = 0`: 8,3 %). En los escenarios alto y bajo se repite el patrón: la logística erra un −28 % al año y Gompertz, entre −10 % y −11 %. Es la asimetría ya observada en 2.4 (`ν < 1` en las series nórdicas): crecimiento rápido al principio y aproximación lenta al techo.
- **Limitación.** La proyección no satura dentro de la ventana: sin bajas, una vez que la cuota llega a su techo el stock crece de forma lineal hasta que se renueva todo el parque (unos 21 años). Por eso el `K` ajustado es una aproximación válida dentro de la ventana de 10 años, y la ventana tiene que coincidir con el horizonte `TF` de la simulación.

#### 2.5.5. Comparación y elección

In [ ]:
# Comparación de las estrategias sobre el horizonte.
display(pd.DataFrame([evaluar_curva(e, P) for e, (P, _) in evaluaciones.items()]).set_index("estrategia")
        .map(lambda v: f"{v:.3g}"))

CURVAS_GRAFICO = ["B · PS 30% V2", "B · PS 30% V1", "C · analogía Mediana", "D · proyección base",
                  "D · logística base", "D · Gompertz base", "D · logística alto", "D · logística bajo"]
t_hist = (caba["anio_stock"] - 2020).to_numpy()
t_fut = np.linspace(T_INICIO, T_INICIO + HORIZONTE_ANIOS, 400)
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
for i, estrategia in enumerate(CURVAS_GRAFICO):
    P = evaluaciones[estrategia][0]
    estilo = "--" if "proyección" in estrategia else "-"
    ejes[0].plot(2020 + t_fut, P(t_fut), estilo, color=PALETA[i], linewidth=1.8, label=estrategia)
    ejes[1].plot(2020 + t_fut, 100 * np.gradient(P(t_fut), t_fut) / COTA_CRECIMIENTO, estilo, color=PALETA[i],
                 linewidth=1.8, label=estrategia)
ejes[0].plot(caba["anio_stock"], caba["V2"], "o", color=COLOR_CABA, markersize=3, label="CABA V2 (observado)")
ejes[0].set_yscale("log")
formato_ejes(ejes[0], "Stock de enchufables que cargan en estaciones (CABA)", "Año", "Vehículos (escala logarítmica)")
ejes[1].axhline(100, color=COLOR_CABA, linewidth=1, linestyle=":")
ejes[1].plot(caba["anio_ventas"], caba["pct_enchufables_ventas"], "o", color=COLOR_CABA, markersize=3,
             label="CABA (observado)")
formato_ejes(ejes[1], "% de enchufables en las ventas implícito en cada curva", "Año", "% de los autos nuevos")
ejes[1].legend().set_visible(False)
plt.tight_layout()
plt.show()

**Elección de los parámetros.**
- Se adopta la **estrategia D, escenario base**. Es la única que, a la vez, coincide en `T = 0` con la cuota observada y respeta la cota física en todo el horizonte. A y B producen curvas imposibles o inconsistentes con el ritmo actual; C lo subestima.
- **Forma.** La logística de la propuesta subestima el crecimiento de los primeros tres años: el factor al año es 0,47, contra 0,34 de la proyección. A partir del quinto año coincide. Gompertz reproduce la proyección con errores de hasta 6 % en el escenario base (hasta 11 % en los otros dos). **Se recomienda pasar a Gompertz**, lo que implica corregir la fórmula de la propuesta. Hasta que el equipo lo decida, el factor de 2.6 usa la logística y el cambio es una línea (`FORMA_CRECIMIENTO = "Gompertz"`).
- **Sensibilidad.** Los escenarios alto (techo de Noruega) y bajo (techo de Costa Rica) quedan disponibles en `PARAMS_CRECIMIENTO`. Las diferencias aparecen después del tercer año: a los 10 años el stock es ×71 en el alto y ×27 en el bajo.
- **Variante.** Se usa `V2`. Con la estrategia D, `V1` daría un resultado casi idéntico, porque su último valor difiere en 2,5 % y los incrementos proyectados son los mismos.

#### 2.5.6. Estrategia E: techo de los países, velocidad y punto de partida de CABA

> **[EXPLORACIÓN]** Alternativa a la estrategia D, en evaluación. Si se adopta, reemplaza la elección de 2.5.5 y los parámetros que usa 2.6, que por ahora siguen siendo los de D.

Es una variante de la estrategia B y retoma el planteo original del equipo: el techo `K` sale de los países de referencia, y la serie de CABA aporta la velocidad y el punto de partida. A diferencia de B, `K` no se obtiene de un potencial de mercado sobre el parque estimado, sino del cociente `K / P(cruce)` de 2.5.3: cuántas veces creció el stock de cada país desde que su cuota de enchufables en los patentamientos cruzó el 9,1 % actual de CABA hasta el techo de su logística. Así, `K` no depende del parque de CABA ni del nivel futuro de los patentamientos.

`K = P(T=0) × K / P(cruce)`. La curva se ancla al stock observado en `T = 0`, lo que fija su parámetro de posición (`A` en la logística, `b` en Gompertz), y la velocidad (`r` o `c`) se ajusta por mínimos cuadrados en niveles sobre la serie `V2` de CABA.

Se comparan cuatro alternativas: la logística y Gompertz, cada una con el techo de Costa Rica o con la mediana de los siete mercados maduros (sin Costa Rica). Las combinaciones descartadas (la variante `V1`, las curvas de Richards y Bass, y la mediana con Costa Rica) y sus motivos se resumen al final de la subsección.

Los controles son los de 2.5: la cuota implícita en `T = 0` contra la observada, y la cuota máxima contra la cota física (100 %) y contra el techo de la cuota en los mercados maduros (mediana de 2.5.4). La cuota implícita usa el nivel de patentamientos de los últimos cuatro trimestres. Como ese supuesto no entra en la curva, la cuota máxima se informa también con el nivel de 2023–2024 (unos 55 000 patentamientos por año).

In [ ]:
# Estrategia E: K = P(T=0) × K/P(cruce) del país de referencia (2.5.3). La curva pasa por el stock observado en T = 0
# (logistica_anclada y gompertz_anclada, de 2.5.4) y la velocidad se ajusta por mínimos cuadrados en niveles sobre la
# serie V2 de CABA, con varios arranques. Son 4 ajustes de 1 parámetro sobre 26 puntos: se corren en serie.
REFERENCIAS_TECHO = {   # cociente K / P(cruce) de 2.5.3
    "Mediana sin Costa Rica": analogias.loc[PAISES_MADUROS, "K / P(cruce)"].median(),
    "Costa Rica": analogias.loc["Costa Rica", "K / P(cruce)"],
}
CURVAS_E = {  # nombre: (curva anclada, arranques de la velocidad)
    "Logística": (logistica_anclada, [0.3, 1.0, 2.0]),
    "Gompertz": (gompertz_anclada, [0.1, 0.3, 1.0]),
}
COTA_VELOCIDAD = 10   # 1/año, la misma cota de 2.4

t_caba = (caba["anio_stock"] - 2020).to_numpy()
y_caba = caba["V2"].to_numpy()
recientes = (caba["anio"] >= 2024).to_numpy()          # del 1.er trimestre de 2024 al 2.º de 2026

def ajustar_E(curva, K):
    """Velocidad por mínimos cuadrados en niveles, con K fijo y varios arranques; devuelve la de menor error."""
    funcion, arranques_E = CURVAS_E[curva]
    mejor = None
    for v0 in arranques_E:
        with warnings.catch_warnings(), np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            warnings.simplefilter("ignore", OptimizeWarning)
            try:
                (v,), _ = curve_fit(lambda t, v: funcion(t, K, v), t_caba, y_caba, p0=[v0],
                                    bounds=([0], [COTA_VELOCIDAD]), maxfev=5000)
            except RuntimeError:      # el arranque no convergió: se prueba el siguiente
                continue
            sse = np.sum((y_caba - funcion(t_caba, K, v)) ** 2)
        if np.isfinite(sse) and (mejor is None or sse < mejor[1]):
            mejor = (v, sse)
    if mejor is None:
        raise RuntimeError(f"{curva}: ningún arranque convergió.")
    return mejor[0]

T_PROY = np.linspace(T_INICIO, T_INICIO + HORIZONTE_ANIOS, 1201)
T_LARGO = np.linspace(0, 60, 6001)                    # inflexión y 95 % de K, aunque caigan fuera del horizonte
COTA_CRECIMIENTO_23_24 = caba.loc[caba["anio"].isin([2023, 2024]), "autos_nuevos"].sum() / 2 * PORC_CARGA_ESTACIONES
ANIOS_E = (1, 3, 4, 5, 10)
COLUMNAS_CRECIMIENTO = [f"× {a} año{'s' if a > 1 else ''}" for a in ANIOS_E]

filas, curvas_E = [], {}
for referencia, cociente in REFERENCIAS_TECHO.items():
    K = P_INICIO * cociente
    for curva, (funcion, _) in CURVAS_E.items():
        v = ajustar_E(curva, K)
        P = lambda t, f=funcion, K=K, v=v: f(t, K, v)
        curvas_E[(referencia, curva)] = P
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            y_aj, p_proy, p_largo = P(t_caba), P(T_PROY), P(T_LARGO)
        pendiente = np.gradient(p_proy, T_PROY)
        llega_95 = p_largo >= 0.95 * K
        filas.append({
            "referencia": referencia, "curva": curva, "K": round(K), "velocidad (1/año)": v,
            "en la cota": bool(np.isclose(v, 0) or np.isclose(v, COTA_VELOCIDAD)),
            **metricas(y_caba, y_aj, 1),
            "MAPE 2024–2026 (%)": 100 * np.mean(np.abs(y_aj[recientes] / y_caba[recientes] - 1)),
            **{columna: P(T_INICIO + a) / P_INICIO for columna, a in zip(COLUMNAS_CRECIMIENTO, ANIOS_E)},
            "inflexión (año)": 2020 + T_LARGO[np.argmax(np.gradient(p_largo, T_LARGO))],
            "95 % de K (año)": 2020 + T_LARGO[np.argmax(llega_95)] if llega_95.any() else np.nan,
            "cuota en T=0": 100 * pendiente[0] / COTA_CRECIMIENTO,
            "cuota máx.": 100 * pendiente.max() / COTA_CRECIMIENTO,
            "cuota máx. (mercado 2023–2024)": 100 * pendiente.max() / COTA_CRECIMIENTO_23_24,
            "cuota a 10 años": 100 * pendiente[-1] / COTA_CRECIMIENTO,
        })

comparacion_E = pd.DataFrame(filas).set_index(["referencia", "curva"])
comparacion_E["ΔAIC"] = comparacion_E["AIC"] - comparacion_E.groupby(level="referencia")["AIC"].transform("min")
print("K / P(cruce) por referencia: " + " | ".join(f"{r}: {v:.2f}" for r, v in REFERENCIAS_TECHO.items()))
print("Ajuste a la serie V2 de CABA. MAPE en %; ΔAIC entre las dos curvas con el mismo techo (0 = mejor).")
display(comparacion_E[["K", "velocidad (1/año)", "en la cota", "MAPE (%)", "MAPE 2024–2026 (%)", "ΔAIC"]]
        .round({"velocidad (1/año)": 3, "MAPE (%)": 1, "MAPE 2024–2026 (%)": 1, "ΔAIC": 1}))
print("Proyección desde T = 0 (stock relativo al de T = 0) y controles. Cuotas: % de enchufables en los patentamientos "
      f"implícito en la curva; observado en T = 0: {PCT_ENCHUFABLES_CABA:.1f} %; techo de la cuota en los mercados "
      f"maduros: {cuotas_ref.loc['Mediana maduros', 'S_MAX (%)']:.1f} %.")
display(comparacion_E[COLUMNAS_CRECIMIENTO + ["inflexión (año)", "95 % de K (año)", "cuota en T=0", "cuota máx.",
                                             "cuota máx. (mercado 2023–2024)", "cuota a 10 años"]].round(1))

In [ ]:
# Stock proyectado y cuota implícita de las cuatro alternativas. Color: curva; trazo: referencia del techo.
COLOR_CURVA = {curva: PALETA[i] for i, curva in enumerate(CURVAS_E)}
TRAZO_REFERENCIA = {"Mediana sin Costa Rica": "-", "Costa Rica": "--"}
S_MAX_MADUROS = cuotas_ref.loc["Mediana maduros", "S_MAX (%)"]
t_graf = np.linspace(t_caba[0], T_INICIO + HORIZONTE_ANIOS, 500)

fig, ejes = plt.subplots(1, 2, figsize=(15, 5.5))
for (referencia, curva), P in curvas_E.items():
    estilo = dict(linestyle=TRAZO_REFERENCIA[referencia], color=COLOR_CURVA[curva], linewidth=1.8,
                  label=f"{curva} · {referencia}")
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        ejes[0].plot(2020 + t_graf, P(t_graf), **estilo)
        ejes[1].plot(2020 + T_PROY, 100 * np.gradient(P(T_PROY), T_PROY) / COTA_CRECIMIENTO, **estilo)
ejes[0].plot(caba["anio_stock"], caba["V2"], "o", color=COLOR_CABA, markersize=3, label="CABA V2 (observado)")
ejes[0].set_ylim(bottom=0)
formato_ejes(ejes[0], "Stock de enchufables que cargan en estaciones (CABA)", "Año", "Vehículos")
ejes[1].plot(caba["anio_ventas"], caba["pct_enchufables_ventas"], "o", color=COLOR_CABA, markersize=3,
             label="CABA (observado)")
ejes[1].axhline(100, color=COLOR_CABA, linewidth=1, linestyle=":", label="Cota física")
ejes[1].axhline(S_MAX_MADUROS, color=COLOR_CABA, linewidth=1, linestyle="-.", label="Techo de la cuota (maduros)")
formato_ejes(ejes[1], "% de enchufables en los patentamientos implícito en cada curva", "Año", "% de los patentamientos")
for ax in ejes:
    ax.axvline(2020 + T_INICIO, color="#999999", linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Factor sobre el IA, P(T=0) / P(T), de las cuatro alternativas, contra el de la proyección de la estrategia D (base).
T_anios = np.linspace(0, HORIZONTE_ANIOS, 400)
t_proy_D, stock_proy_D = proyecciones["base"]
fig, ax = plt.subplots(figsize=(10, 5))
for (referencia, curva), P in curvas_E.items():
    ax.plot(2020 + T_INICIO + T_anios, P_INICIO / P(T_INICIO + T_anios), linestyle=TRAZO_REFERENCIA[referencia],
            color=COLOR_CURVA[curva], linewidth=1.8, label=f"{curva} · {referencia}")
ax.plot(2020 + t_proy_D, P_INICIO / stock_proy_D, ":", color=COLOR_CABA, linewidth=1.4, label="Proyección D (base)")
ax.set_ylim(bottom=0)
formato_ejes(ax, "Factor sobre el IA de las cuatro alternativas", "Año", "Factor sobre el IA")
plt.tight_layout()
plt.show()

**Análisis de la estrategia E.**

*Descartes.* Primero se corrió la grilla completa: variantes `V1` y `V2`; techo de Costa Rica, mediana sin Costa Rica y mediana con Costa Rica; logística, Gompertz, Richards y Bass. Son 24 combinaciones, que se recortaron a las cuatro alternativas de arriba. Los motivos de los descartes:
- **`V1`.** En las 12 combinaciones, la cuota implícita en `T = 0` queda entre 3,2 % y 4,2 %, contra el 9,1 % observado. La serie arrastra el ritmo del mercado electrificado total, porque aplica el 18,3 % de enchufables también a 2020–2021, cuando eran el 1 % (2.2). Su menor error sobre los datos recientes (MAPE de 7 % a 12 %) se debe a que la serie es más suave, no a que sea más correcta.
- **Richards.** El parámetro de forma se va a `ν` = 6–10 y la velocidad a `r` = 4–10; en `V2` con el techo de Costa Rica, `r` queda en la cota. La curva acelera hasta el último momento y frena de golpe: con `V2`, llega al 95 % de `K` entre 2028 y 2029. Con `V2` y las medianas exige que entre el 176 % y el 220 % de los patentamientos sean enchufables, lo que es físicamente imposible. Sólo mejora el AIC con el techo de Costa Rica (4,7 puntos sobre la logística en `V2`); con las medianas empata con la logística. La serie de CABA no identifica la forma, y el ajuste elige la más extrema.
- **Bass.** En `V2`, el anclaje deja `p` del orden de 10⁻⁶ y Bass coincide con la logística en toda la proyección, con diferencias de ajuste de 1 a 1,5 puntos de MAPE. En `V1` difiere poco. No aporta nada que no dé la logística, y agrega un supuesto: el origen de la adopción en 2020.
- **Mediana con Costa Rica, como referencia base.** Cuenta dos veces a Costa Rica, en la base y en el escenario bajo, y mezcla en la mediana un país que no completó la transición (17 % de enchufables en las ventas de 2025). Sus resultados quedan entre los de las otras dos referencias.

*Alternativas (todas con `V2`).*

1. **Gompertz con la mediana sin Costa Rica.**
   - A favor: reproduce el ritmo actual (cuota de 9,0 % en `T = 0`). La cuota máxima es de 31 % (47 % con el mercado de 2023–2024), siempre por debajo del techo de los mercados maduros (63,7 %). El crecimiento es gradual: ×2,9, ×10,2, ×14,6, ×18,7 y ×30,3 a 1, 3, 4, 5 y 10 años. A los 10 años todavía suma enchufables (7 % de los patentamientos).
   - En contra: ajusta peor la serie de CABA (MAPE 2024–2026 de 30 %, contra 15 % de la logística; ΔAIC de 27). Supone que CABA ya empieza a frenar mientras los datos todavía aceleran, y subestima el stock de 2020–2023 en varios órdenes de magnitud. Exige cambiar la fórmula de la propuesta.
2. **Gompertz con el techo de Costa Rica.**
   - A favor: es la más conservadora. La cuota máxima es de 12 % (18 % con el mercado de 2023–2024) y el crecimiento es de ×2,5, ×5,7, ×6,8, ×7,5 y ×8,6 a 1, 3, 4, 5 y 10 años. Reproduce el ritmo actual (8,3 %). Es el escenario bajo natural de la alternativa 1: misma curva, techo del único par latinoamericano que ya muestra el suyo.
   - En contra: es el peor ajuste a la serie de CABA (MAPE 2024–2026 de 36 %; ΔAIC de 30). El techo sale de un solo país, estimado con pocos años de desaceleración. Exige cambiar la fórmula de la propuesta.
3. **Logística con el techo de Costa Rica.**
   - A favor: es la fórmula de la propuesta. Ajusta bien la serie (MAPE 2024–2026 de 17 %), reproduce el ritmo actual (10,1 %) y pasa los controles: la cuota máxima es de 25 % (37 % con el mercado de 2023–2024).
   - En contra: el techo sale de un solo país. El crecimiento total es de ×8,7 y se completa en tres años (95 % de `K` en 2029,6). Desde ahí la demanda queda plana y la curva deja de sumar enchufables (cuota de 0 % a los 10 años): toda la "S" ocurre en los primeros tres años del horizonte.
4. **Logística con la mediana sin Costa Rica.**
   - A favor: es la fórmula de la propuesta y la de mejor ajuste a la serie de CABA (MAPE 2024–2026 de 15 %). Reproduce el ritmo actual (10,5 %), y el techo sale de siete países.
   - En contra: la velocidad ajustada, `r` = 1,53/año, casi triplica la mediana de los países (0,55/año), y el stock llega al 95 % del techo en 2030,7. En 2028 exige una cuota del 90 % de los patentamientos (134 % con el mercado de 2023–2024), por encima del techo de los mercados maduros y, en el segundo caso, de la cota física. Después la cuota cae a 0 %. Ajustar mejor el pasado no la favorece para proyectar: en el tramo observado todas las curvas aceleran, y sólo se diferencian en cómo frenan.

*Factor sobre el IA.* Gompertz con la mediana sigue de cerca a la proyección de D hasta 2028 (×10,2 contra ×11,7 a los 3 años) y después se aplana, mientras que D sigue creciendo porque no satura dentro de la ventana. La logística con la mediana crece más rápido que D en los primeros años y se aplana hacia 2030. Con el techo de Costa Rica, las dos curvas terminan en el mismo nivel (×8,7): la logística llega en 2029 y Gompertz, hacia 2033.

> **[DECISIÓN ABIERTA]** Recomendación provisoria: alternativa 1, con la 2 como escenario bajo. Si el equipo prefiere conservar la fórmula de la propuesta, la alternativa 3 es la única logística que pasa los controles. La elección define los parámetros de 2.6 y la corrección de la propuesta.

### 2.6. Factor de crecimiento aplicado al Intervalo entre Arribos

Función que, dado el instante `T` de la simulación (minutos desde el inicio), devuelve el factor que multiplica el `IA` generado: menor a 1 en la fase de aceleración y mayor a 1 en la de desaceleración.

> **[DECISIÓN ABIERTA]** La propuesta pide un factor que achique el IA en la fase de aceleración y lo agrande en la de desaceleración. Eso sólo ocurre si el factor se calcula sobre la **tasa** de crecimiento `P'(t)`, que tiene forma de campana. Sobre el **stock** `P(t)`, el factor es siempre decreciente: el IA se achica rápido durante la aceleración y deja de achicarse al saturar, pero nunca vuelve a agrandarse.
>
> Se adopta el factor sobre el stock, `P(T=0) / P(T)`: los arribos a las estaciones los generan todos los autos enchufables en circulación, no sólo los que se incorporan en ese momento. Con el factor sobre `P'(t)`, la demanda tendería a cero cuando el parque se satura. Las dos versiones se grafican abajo. Si el equipo confirma la elección, hay que corregir la redacción de la propuesta y la de esta subsección.

In [ ]:
# Parámetros adoptados: estrategia D (2.5.4). K en vehículos, tasas en 1/año, t en años desde el 1/1/2020.
# FORMA_CRECIMIENTO: "logística" (forma de la propuesta) o "Gompertz" (mejor ajuste a la proyección, ver 2.5.5).
FORMA_CRECIMIENTO = "logística"
ESCENARIO_CRECIMIENTO = "base"
PARAMS_CRECIMIENTO = {(forma, escenario): evaluaciones[f"D · {forma} {escenario}"][1]
                      for forma in ("logística", "Gompertz") for escenario in ESCENARIOS_CUOTA}
K_LOG, A_LOG, R_LOG = PARAMS_CRECIMIENTO[("logística", ESCENARIO_CRECIMIENTO)]   # K, A y r de la propuesta
MIN_POR_ANIO = 12 * 30 * 24 * 60    # 518 400 min: año de 12 meses de 30 días, la misma convención que TCE
PARAMS_ADOPTADOS = PARAMS_CRECIMIENTO[(FORMA_CRECIMIENTO, ESCENARIO_CRECIMIENTO)]

def P_crecimiento(t, forma=FORMA_CRECIMIENTO, params=PARAMS_ADOPTADOS):
    """Stock de enchufables que cargan en estaciones en CABA; t en años desde el 1/1/2020."""
    K, a, b = params
    if forma == "logística":
        return K / (1 + a * math.exp(-b * t))      # K / (1 + A·e^(−r·t))
    return K * math.exp(-a * math.exp(-b * t))     # K·exp(−b·e^(−c·t))

def factor_crecimiento(T, forma=FORMA_CRECIMIENTO, params=PARAMS_ADOPTADOS):
    """Factor que multiplica el IA generado en el instante T (min desde el inicio de la simulación).
    Es P(T = 0) / P(T): vale 1 al inicio y decrece hacia P(T = 0) / K a medida que el parque satura."""
    return P_crecimiento(T_INICIO, forma, params) / P_crecimiento(T_INICIO + T / MIN_POR_ANIO, forma, params)

print(f"Adoptado: forma {FORMA_CRECIMIENTO}, escenario {ESCENARIO_CRECIMIENTO}")
display(pd.DataFrame([{"forma": forma, "escenario": escenario, "K": K, "A (logística) / b (Gompertz)": a,
                       "r (logística) / c (Gompertz), 1/año": b}
                      for (forma, escenario), (K, a, b) in PARAMS_CRECIMIENTO.items()]).set_index(["forma", "escenario"])
        .round(4))

# Chequeos sobre el factor (valores, no ausencia de excepciones), para todas las formas y escenarios.
T_grilla = np.linspace(0, HORIZONTE_ANIOS * MIN_POR_ANIO, 500)
for (forma, escenario), params in PARAMS_CRECIMIENTO.items():
    f = np.array([factor_crecimiento(T, forma, params) for T in T_grilla])
    assert math.isclose(P_crecimiento(T_INICIO, forma, params), P_INICIO), f"{forma} {escenario}: no pasa por P_INICIO."
    assert math.isclose(f[0], 1.0), f"{forma} {escenario}: el factor no vale 1 en T = 0."
    assert np.all(np.diff(f) < 0), f"{forma} {escenario}: el factor no es estrictamente decreciente."
    limite = P_INICIO / params[0]
    assert math.isclose(factor_crecimiento(1e12, forma, params), limite, rel_tol=1e-9), f"{forma} {escenario}: no tiende a P(T=0)/K."
    assert f[-1] > limite, f"{forma} {escenario}: el factor cruzó su límite inferior."

# Contraste con la proyección física del stock (estrategia D): P(T=0) / P(T) de cada forma vs. de la proyección.
filas = []
for escenario in ESCENARIOS_CUOTA:
    t_proy, stock_proy = proyecciones[escenario]
    for anios in (1, 3, 5, 10):
        fila = {"escenario": escenario, "T (años)": anios,
                "proyección": P_INICIO / np.interp(T_INICIO + anios, t_proy, stock_proy)}
        for forma in ("logística", "Gompertz"):
            fila[forma] = factor_crecimiento(anios * MIN_POR_ANIO, forma, PARAMS_CRECIMIENTO[(forma, escenario)])
        filas.append(fila)
display(pd.DataFrame(filas).set_index(["escenario", "T (años)"]).round(3))

In [ ]:
# Alternativa no adoptada (lectura literal de la propuesta): factor sobre la tasa P'(t) en lugar del stock.
def factor_tasa(T, forma=FORMA_CRECIMIENTO, params=PARAMS_ADOPTADOS):
    """P'(T = 0) / P'(T): supone arribos proporcionales a las altas de autos, no al stock en circulación."""
    K, a, b = params
    def derivada(t):
        if forma == "logística":
            e = a * math.exp(-b * t)
            return K * b * e / (1 + e) ** 2
        e = math.exp(-b * t)
        return K * a * b * e * math.exp(-a * e)
    return derivada(T_INICIO) / derivada(T_INICIO + T / MIN_POR_ANIO)

anios_grilla = T_grilla / MIN_POR_ANIO
fig, ejes = plt.subplots(1, 3, figsize=(17, 4.5))
t_proy, stock_proy = proyecciones[ESCENARIO_CRECIMIENTO]
ejes[0].plot(2020 + t_proy, P_INICIO / stock_proy, "o", color=COLOR_CABA, markersize=3, label="Proyección (estrategia D)")
for i, forma in enumerate(("logística", "Gompertz")):
    params = PARAMS_CRECIMIENTO[(forma, ESCENARIO_CRECIMIENTO)]
    ejes[0].plot(2026.5 + anios_grilla, [factor_crecimiento(T, forma, params) for T in T_grilla], color=PALETA[i],
                 linewidth=1.8, label=forma.capitalize())
formato_ejes(ejes[0], f"Factor por forma (escenario {ESCENARIO_CRECIMIENTO})", "Año", "Factor sobre el IA (escala logarítmica)")
for i, escenario in enumerate(ESCENARIOS_CUOTA):
    params = PARAMS_CRECIMIENTO[(FORMA_CRECIMIENTO, escenario)]
    ejes[1].plot(2026.5 + anios_grilla, [factor_crecimiento(T, FORMA_CRECIMIENTO, params) for T in T_grilla],
                 color=PALETA[i], linewidth=1.8, label=f"Escenario {escenario}")
formato_ejes(ejes[1], f"Factor por escenario ({FORMA_CRECIMIENTO})", "Año", "Factor sobre el IA (escala logarítmica)")
ejes[2].plot(2026.5 + anios_grilla, [factor_crecimiento(T) for T in T_grilla], color=PALETA[0], linewidth=1.8,
             label="Sobre el stock P(t) (adoptado)")
ejes[2].plot(2026.5 + anios_grilla, [factor_tasa(T) for T in T_grilla], color=PALETA[1], linewidth=1.8,
             linestyle="--", label="Sobre las altas P'(t) (lectura literal)")
ejes[2].axhline(1, color=COLOR_CABA, linewidth=1, linestyle=":")
formato_ejes(ejes[2], "Factor sobre el stock vs. sobre las altas", "Año", "Factor sobre el IA (escala logarítmica)")
for ax in ejes:
    ax.set_yscale("log")
plt.tight_layout()
plt.show()

**Factor adoptado** (logística, escenario base): `factor_crecimiento(T)` vale 0,47 a 1 año, 0,11 a 3, 0,036 a 5 y 0,015 a 10. Es decir, el flujo de arribos de cada estación se multiplica por 2,1, 8,8, 28 y 68 respecto de `T = 0`. Con Gompertz los valores son 0,36, 0,086, 0,037 y 0,015. La función es escalar (`math.exp`), está definida a nivel de módulo para que sea serializable en los procesos del barrido, y recibe la forma y los parámetros como argumentos para poder correr los escenarios de sensibilidad.

`T = 0` supone que las FDPs de IA ajustadas en la sección 5 representan la demanda de una estación al 1/7/2026. Ese supuesto se explicita también al ajustarlas.

## 3. Carga de Datos Inicial

En esta sección se carga el dataset para la generación de las Funciones de Densidad de Probabilidad (FDP)

> **[DESACTUALIZADO — TP 4]** La ruta y el archivo son los del TP 4. El TP Final usa `Dataset California.csv` en `/content/drive/MyDrive/Colab Notebooks/TP Final Simu/Datos/`.

In [ ]:
# [DESACTUALIZADO — TP 4] actualizar PATH_DATOS y el nombre del dataset.
# Ruta de acceso a los datos
PATH_DATOS = "/content/drive/MyDrive/Colab Notebooks/TP 4 Simu/Datos/"

try:
    # Carga de datasets para el análisis de variables estocásticas
    df = pd.read_csv(PATH_DATOS + "EVChargingStationUsage.csv")

    print("Dataset inicializado correctamente.")
except FileNotFoundError as e:
    print(f"Error en la carga de recursos: {e}")

## 4. Obtención de la FDP del Tiempo de Carga (TC)

El Tiempo de Carga representa la duración de la sesión de carga. Se utiliza la columna `Charging Time (hh:mm:ss)` del dataset mapeada a minutos para determinar la distribución estocástica que mejor represente este proceso, y poder generar valores aleatorios para dicha distribución.

> **[DESACTUALIZADO — TP 4]** Reajustar la FDP de `TC` sobre el dataset del TP Final. De este ajuste salen `TCP` (`E[TC]` truncado) y `FI = 41,7 / TCP`; no se reciclan los valores del TP 4.

In [ ]:
# [DESACTUALIZADO — TP 4] reejecutar con el dataset del TP Final.
def mostrar_histograma(data, title, xlabel):
    plt.figure(figsize=(7,5))
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Densidad de Probabilidad")
    plt.hist(data, bins=100, density=True)
    plt.show()

df["Charging Time (minutes)"] = (
    pd.to_timedelta(df["Charging Time (hh:mm:ss)"].dropna())
    .dt.total_seconds() / 60
)

mostrar_histograma(df["Charging Time (minutes)"], "Histograma del Tiempo de Carga", "Tiempo de Carga (minutos)")

Dado que la distribución tiene una cola derecha larga, se verá las distribuciónes que mejor se ajusten usando como criterio el `ks_statistic`.

In [ ]:
# [DESACTUALIZADO — TP 4] reejecutar con el dataset del TP Final.
def mostrar_mejores_dist(data, nombre, criterio):
    """Ajusta y visualiza las mejores distribuciones para una serie de datos."""
    print(f"--- Ajustando distribuciones para {nombre} según {criterio} ---")
    f = Fitter(data)
    f.fit()
    display(f.summary(method=criterio))
    return f

f_tc = mostrar_mejores_dist(df["Charging Time (minutes)"], "Tiempo de Carga", "ks_statistic")

Se selecciona la distribución `Gumbel R`, pues tiene el ks_statistic más bajo, y también gana en sumsquare_error y kl_div.

In [ ]:
# [DESACTUALIZADO — TP 4] reejecutar con el dataset del TP Final.
def mostrar_fdp(title, xlabel, f, fdp, data):
    plt.figure(figsize=(7,5))
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Densidad de Probabilidad")
    plt.hist(data, bins=100, density=True, alpha=0.5)
    f.plot_pdf(names=[fdp])
    plt.show()

def obtener_dist(f, funcion, nombre, data):
    """Extrae una distribución scipy lista para usar a partir de un Fitter ajustado."""
    if funcion not in f.fitted_param:
        disponibles = list(f.fitted_param.keys())
        raise ValueError(f"'{funcion}' no fue ajustada. Disponibles: {disponibles}")

    params_tuple = f.fitted_param[funcion]
    dist_class   = getattr(stats, funcion)
    distribucion = dist_class(*params_tuple)

    param_names  = (dist_class.shapes.split(", ") if dist_class.shapes else []) + ["loc", "scale"]
    params_dict  = dict(zip(param_names, params_tuple))

    print(f"La función elegida para {nombre} es: {funcion}")
    print(f"Con los parámetros: {params_dict}")
    print(f"Acotada entre: {data.min()} y {data.max()}")

    mostrar_fdp(f"FDP para {nombre}", f"{nombre} (minutos)", f, funcion, data)

    return distribucion

dist_tc = obtener_dist(f_tc, "gumbel_r", "Tiempo de Carga", df["Charging Time (minutes)"])

def rvs_truncado(dist, data):
    """
    Genera muestras de una distribución truncada por los límites originales
    de los datos, sin acumular probabilidad en los bordes.
    """
    low  = data.min()
    high = data.max()
    p_low  = dist.cdf(low)    # probabilidad acumulada hasta el límite inferior
    p_high = dist.cdf(high)   # probabilidad acumulada hasta el límite superior

    # Muestreamos uniforme SOLO en la franja de probabilidad válida
    u = np.random.uniform(p_low, p_high)

    # Transformada inversa: convierte probabilidades → valores
    return dist.ppf(u)

def generar_tc():
    """Generador estocástico de Tiempo de Carga para el motor de simulación."""
    return float(rvs_truncado(dist_tc, df["Charging Time (minutes)"]))

### 4.1. Tiempo de carga promedio (TCP) y factor de impaciencia (FI)

In [ ]:
# [PENDIENTE] TCP = E[TC] de la FDP truncada al rango observado; FI = 41.7 / TCP.

## 5. Obtención de las FDPs del Intervalo entre Arribos (IA)

El Intervalo de Arribos representa el tiempo entre que llega un vehículo al sistema y llega otro. Se utiliza la columna `Start Date` del dataset para calcular el tiempo entre llegadas, determinar la distribución estocástica que mejor represente este proceso, y poder generar valores aleatorios para dicha distribución.

> **[DESACTUALIZADO — TP 4]** El TP 4 ajusta un único `IA` global. El TP Final requiere seis FDPs (`IAS1`, `IAS2`, `IAS3`, `IAF1`, `IAF2`, `IAF3`) por franja horaria y tipo de día. Las celdas siguientes quedan como referencia del pipeline.

In [ ]:
# [DESACTUALIZADO — TP 4] IA global; reemplazar por las seis FDPs por franja.
df["Start Date"] = pd.to_datetime(df["Start Date"].dropna(), format="%m/%d/%Y %H:%M")
df = df.sort_values("Start Date")
df["Interarrival Time"] = df["Start Date"].diff().dt.total_seconds() / 60
df["Interarrival Time"] = df["Interarrival Time"].replace(0, 0.5)

data_arribos = df["Interarrival Time"].dropna()

mostrar_histograma(data_arribos, "Histograma del Intervalo de Arribos", "Intervalo de Arribos (minutos)")

Dado que esta distribución también tiene una cola derecha muy larga, se verá las distribuciónes que mejor se ajusten usando como criterio el `ks_statistic`.

In [ ]:
# [DESACTUALIZADO — TP 4] IA global; reemplazar por las seis FDPs por franja.
f_ia = mostrar_mejores_dist(data_arribos, "Intervalo de Arribos", "ks_statistic")

Se selecciona la distribución `Landau`, pues por más que no sea la de menor ks_statistic, no está lejos, y tiene menor sumsquare_error y menor kl_div, los cuales también son importantes de analizar para Simulación.

In [ ]:
# [DESACTUALIZADO — TP 4] IA global; reemplazar por las seis FDPs por franja.
dist_ia = obtener_dist(f_ia, "landau", "Intervalo de Arribos", data_arribos)

def generar_ia():
    """Generador estocástico de Intervalo de Arribos para el motor de simulación."""
    return float(rvs_truncado(dist_ia, data_arribos))

### 5.1. Segmentación por franja horaria y tipo de día

Franjas: 08:00–12:59, 13:00–19:59 y 20:00–07:59; días de semana y fin de semana.

In [ ]:
# [PENDIENTE] Segmentar los intervalos entre arribos por franja (según la hora del arribo)
# y por tipo de día, obteniendo las seis series IAS1..IAS3 e IAF1..IAF3.

### 5.2. Ajuste en paralelo de las seis FDPs

In [ ]:
# [PENDIENTE] Ajustar las seis series en paralelo (ProcessPoolExecutor, un worker por serie)
# con el mismo pipeline de fitter, y reportar ks_statistic, sumsquare_error y kl_div de cada una.

**Elección de las FDPs:** *[PENDIENTE] Una justificación por cada una de las seis series.*

## 6. Obtención de las FDPs de Falla (TFC) y Reparación (TRC) de Cargadores

`TFC` es el tiempo desde la puesta en servicio (reparación) de un cargador hasta su falla siguiente; `TRC`, el tiempo que permanece fuera de servicio hasta su reparación.

> **[PENDIENTE]** El dataset del TP 4 no contiene estos datos. Falta definir la fuente; si no se consigue un dataset, se adopta una FDP supuesta y se explicita la justificación.

In [ ]:
# [PENDIENTE] Carga de datos / ajuste de TFC y TRC y sus generadores.

## 7. Obtención del Costo Energético en función del TC

El Costo Energético representa el dinero gastado en la energía de los cargadores. Este depende de:
- El Tiempo de Carga (TC)
- El Precio de la Energía (PE): Valor constante que representa cuanto vale el kWh en CABA en Pesos

Para calcularlo, usaremos regresión lineal para obtener una función que nos permita pasar de TC a CE.


> **[DESACTUALIZADO — TP 4]** La regresión del TP 4 ordena por separado tiempo y energía (`np.sort`), lo que no es una regresión sobre los pares reales e infla el R². Rehacerla sobre los pares `(TC_k, kWh_k)` alineados y reportar el R² verdadero. El precio único `PE` se reemplaza por tarifas por franja (pico/valle). Además, la sigla `CE` del TP 4 (costo energético) choca con `CE` de la propuesta (cantidad de estaciones): renombrar.

In [ ]:
# [DESACTUALIZADO — TP 4] regresión sobre series ordenadas por separado; rehacer sobre pares alineados.
PE = 108.48 # Precio Promedio de Energia en CABA, en Pesos

tiempo = df["Charging Time (minutes)"].values
energia = df['Energy (kWh)'].dropna().values

slope, intercept, r, p, se = stats.linregress(
    np.sort(tiempo),
    np.sort(energia)
)

print(f"Energía [kWh] = {slope:.4f} x Tiempo [min] + {intercept:.4f}  (R²={r**2:.4f})")
print(f"Costo [ARS] = Energía [kWH] x {PE} [ARS/kWH]")

def tiempo_a_costo_lineal(t):
    return float((slope * t + intercept) * PE)

### 7.1. Tarifas de energía por franja horaria

In [ ]:
# [PENDIENTE] Precio de la energía por franja (pico/valle) con su fuente o supuesto explicitado.

## 8. Constantes y Parámetros del Modelo

Constantes según la propuesta: `CC_MAX`, `CE_MAX`, `CCP`, `CPN`, `CEN`, `TIC`, `TCE`, `PUNEN`, `FI`, `TCP`; y valores por defecto de las variables de control `RC`, `TMP` y `PDCE`.

> **[PENDIENTE]** Faltan los valores de `CC_MAX`, `CE_MAX`, `CPN`, `CEN`, `CCP` y los costos de mantenimiento. Cada supuesto se explicita con su justificación.

In [ ]:
# [PENDIENTE] Definición de constantes y valores por defecto de las variables de control.

## 9. Funciones Auxiliares

Reloj (mapeo de `T` a hora del día y tipo de día), generación del `IA` con la FDP vigente y el factor de crecimiento, prorrateo de acumuladores y costo de energía entre franjas, y decisión de arrepentimiento (`PUNEN`, `PU`, `EEU`).

In [ ]:
# [PENDIENTE] Funciones auxiliares del motor (a nivel de módulo, para que sean serializables).

def franja_y_tipo_dia(T):
    raise NotImplementedError("Pendiente: mapeo de T a franja y tipo de día (sección 9).")

def generar_ia(T):
    raise NotImplementedError("Pendiente: IA con la FDP vigente y el factor de crecimiento (sección 9).")

## 10. Implementación del Motor de Simulación

Se define un modelo de simulación de eventos discretos para un sistema multi-servidor (N y N) con lógica de abandono (arrepentimiento). El sistema gestiona una lista de eventos futuros basada en arribos y salidas por cada puesto.

> **[DESACTUALIZADO — TP 4]** Motor del TP 4: un solo IA global, elección aleatoria de puesto, arrepentimiento hardcodeado y estado en variables globales. Queda como referencia de la estructura del loop; se reemplaza por el motor de las subsecciones 10.1–10.3.

In [ ]:
# [DESACTUALIZADO — TP 4] motor de referencia; no usar para el TP Final.
# Variables de Control
N: int                        # Cantidad de puestos

#  Estado del reloj y eventos
T: float                      # Reloj actual
TF: float                     # Horizonte de simulación
TPLL: float                   # Próxima llegada global
TPS: List[float]              # Próxima salida por puesto

#  Estado del sistema
NS: List[int]                 # Autos en cada puesto (cola + servicio)

#  Contadores y acumuladores
CLL: int                      # Llegadas efectivas
CARR: int                     # Arrepentimientos
SPS: float                    # suma ponderada de personas en sistema
STC: float                    # Suma de tiempos de carga

ITO: List[float]              # Instante último cambio a ocioso por puesto
STO: List[float]              # Tiempo ocioso acumulado por puesto
SCE: List[float]              # Costo Energético acumulado por puesto

#  Resultados
PPS: float                    # Promedio de personas en sistema
PEC: float                    # Promedio en cola
PARR: float                   # % arrepentimiento
PTO: List[float]              # Proporción de ocio por puesto
CCE: List[float]              # Costo de Consumo Energetico por puesto

# Factor de comportamiento: probabilidad de que un usuario abandone la cola
PORCENTAJE_ARREPENTIMIENTO = 0.93

def condiciones_iniciales(cant_puestos: int):
    global TF, N
    global T, TPLL, TPS
    global NS
    global CLL, CARR, SPS, STC, ITO, STO, SCE

    # Variables de Control
    N = int(cant_puestos)

    # Reloj/eventos
    T = 0.0
    TF = 1000000.0
    TPLL = 0.0

    # Acumuladores
    CLL = 0
    CARR = 0
    SPS = 0.0
    STC = 0.0

    # Vectores
    TPS = [float("inf")] * N
    NS = [0] * N
    ITO = [0.0] * N
    STO = [0.0] * N
    SCE = [0.0] * N

def arrepentimiento(index_puesto) -> bool:
    global CARR, NS, PORCENTAJE_ARREPENTIMIENTO
    cant_autos = NS[index_puesto]

    if cant_autos <= 1:
        return False
    elif cant_autos <= 2:
        r = random.random()
        if r >= PORCENTAJE_ARREPENTIMIENTO:
            return False
        else:
            CARR += 1
            return True
    else:
        CARR += 1
        return True

def llegada():
    global TPLL, SPS, T, NS, CLL, STC, STO, ITO, TPS, N

    SPS += (TPLL - T) * (sum(NS))

    T = TPLL #Avanzamos en el tiempo

    IA = generar_ia() #Generamos el tiempo hasta la proxima llegada
    TPLL = T + IA #Calculamos el tiempo de llegada del proximo auto

    index_puesto = eleccion_server() #Subrutina
    if arrepentimiento(index_puesto):
        return

    # Llegada efectiva
    NS[index_puesto] += 1
    CLL += 1

    if NS[index_puesto] == 1:
        TC = generar_tc()
        STC += TC
        SCE[index_puesto] += tiempo_a_costo_lineal(TC)
        TPS[index_puesto] = T + TC
        STO[index_puesto] += (T - ITO[index_puesto])



def salida(index_puesto):
    global SPS, T, NS, STC, ITO, STO, TPS

    SPS += (TPS[index_puesto] - T) * (sum(NS))

    T = TPS[index_puesto]  # Avanzamos en el tiempo

    NS[index_puesto] -= 1  # Actualizo vector estado

    if NS[index_puesto] > 0:  # Evento Futuro Condicionado
        TC = generar_tc()
        STC += TC
        SCE[index_puesto] += tiempo_a_costo_lineal(TC)
        TPS[index_puesto] = T + TC  # Actualizo vector de tiempos de salida
    else:
        ITO[index_puesto] = T
        TPS[index_puesto] = float("inf")  # Actualizo vector de tiempos de salida

def eleccion_server():
    global N
    return random.randrange(N) # Distribución uniforme de la demanda

def menor_tps_index():
    global TPS
    return min(range(len(TPS)), key=TPS.__getitem__)

def calculo_resultados():
    global SPS, STC, SCE, CLL, CARR, STO, T, N
    total_arribos = CLL + CARR

    PPS = SPS / CLL if CLL > 0 else 0.0
    PEC = (SPS - STC) / CLL if CLL > 0 else 0.0
    PARR = CARR * 100 / total_arribos if total_arribos > 0 else 0.0

    PTO = []
    CCE = []

    for i in range(N):
        PTO.append(STO[i] * 100 / T if T > 0 else 0.0)

    for i in range(N):
        CCE.append(SCE[i] * 100 / CLL if CLL > 0 else 0.0)

    print(f"PPS = {round(PPS,2)}")
    print(f"PEC = {round(PEC,2)}")
    print(f"PTO = {[round(x, 2) for x in PTO]}")
    print(f"CCE = {[round(x, 2) for x in CCE]}")
    print(f"PARR = {round(PARR,2)}")


def simular(cant_puestos: int):
    global TPLL, TPS, T, TF, NS

    condiciones_iniciales(cant_puestos)

    while True:
        index_server = menor_tps_index()

        if TPLL <= TPS[index_server]:
            llegada()
        else:
            salida(index_server)

        if T <= TF:
            continue
        else: # Vaciamiento
            if any(x > 0 for x in NS):
                TPLL = float("inf")
                continue
            break
    calculo_resultados()

### 10.1. Estado de la simulación

In [ ]:
# [PENDIENTE] Estado encapsulado (dataclass): variables de estado CA(i), CD(i), CC(i), CE;
# TEF TPI(i), TPC(i)(j), TPAE, TPIC(i), TPCE, TPFC(i)(j), TPRC(i)(j), TPMP(i);
# auxiliares CARRUM(i), CPAACUM, CAPF(i), ICC(i)(j), CTC(i)(j) y acumuladores por franja.

### 10.2. Rutinas de eventos

Una rutina por evento de la tabla de eventos y condiciones de la propuesta.

In [ ]:
# [PENDIENTE] Rutinas: Ingreso de auto a estación (i), Carga en cargador (i)(j),
# Análisis de Expansión, Instalación de nuevo cargador (i), Construcción de nueva estación,
# Falla de cargador (i)(j), Reparación de cargador (i)(j), Mantenimiento preventivo (i).

### 10.3. Función de simulación

Función pura: recibe parámetros y semilla, devuelve las métricas de resultado (`BM`, `PTO`, `PEC`, `PPS`, `PARR`, `PDC`, `PAPF` por franja y sus promedios).

In [ ]:
# [PENDIENTE] Loop evento a evento (avance al mínimo de la TEF) y cálculo de resultados.

def simular_tp_final(RC, TMP, PDCE, TF, semilla):
    raise NotImplementedError("Pendiente: motor del TP Final (sección 10.3).")

## 11. Verificación y Validación del Motor

Invariantes y aserciones derivados de la tabla de eventos y condiciones de la propuesta, y validación contra el modelo analítico M/M/c en un escenario reducido.

In [ ]:
# [PENDIENTE] Chequeos sobre valores y cambios de estado (no sobre "no tiró excepción"),
# p. ej. autos cargando = min(CA(i), CD(i)), y comparación contra M/M/c.

## 12. Experimentación y Análisis de Sensibilidad

Se ejecutan múltiples escenarios variando el número de puestos de carga (N) para identificar el punto de equilibrio operativo donde se minimiza el arrepentimiento y el tiempo de espera en cola, manteniendo una utilización eficiente del cargador.

> **[DESACTUALIZADO — TP 4]** El barrido del TP 4 varía la cantidad de puestos `N`. En el TP Final la cantidad de cargadores y estaciones es dinámica; los escenarios varían `RC`, `TMP` y `PDCE`.

In [ ]:
# [DESACTUALIZADO — TP 4] barrido sobre N; reemplazar por la grilla de RC, TMP y PDCE.
# Simulación para configuraciones de 1 a 40 puestos
for i in range(1, 41):
    print(f"\n--- ESCENARIO: N = {i} puestos ---")
    simular(cant_puestos=i)

### 12.1. Grilla de escenarios y réplicas en paralelo

In [ ]:
# [PENDIENTE] Grilla de combinaciones (RC, TMP, PDCE) × réplicas, ejecutada con
# ProcessPoolExecutor (workers según os.cpu_count()), con semilla por corrida derivada
# de una semilla base + índice de combinación/réplica.

### 12.2. Análisis de resultados

In [ ]:
# [PENDIENTE] Tablas y gráficos comparativos de las métricas por escenario, con intervalos de confianza.

## 13. Conclusiones

*[PENDIENTE]*